<a href="https://colab.research.google.com/github/angeruzzi/home-credit-default-risk/blob/main/notebooks/04_auxiliary_tables_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Credit Default Risk: Feature Engineering das Tabelas Auxiliares

## Introdução

### 1. Contexto

A tabela `application_train.csv` representa a aplicação de crédito atual e contém informações cadastrais, financeiras e demográficas disponíveis no momento da solicitação.

Entretanto, o conjunto Home Credit também disponibiliza tabelas com o histórico de relacionamento financeiro dos clientes, incluindo:

- créditos registrados em outras instituições;
- aplicações anteriores realizadas na Home Credit;
- comportamento de pagamento;
- parcelas;
- contratos de cartão de crédito;
- contratos de POS/CASH.

Essas informações permitem criar features relacionadas ao histórico de crédito e ao comportamento financeiro anterior do cliente, como:

- quantidade de contratos;
- valores médios e totais de crédito;
- contratos ativos e encerrados;
- atrasos;
- pagamentos incompletos;
- utilização de limites;
- frequência de aplicações anteriores;
- proporção de propostas aprovadas e recusadas.

O objetivo deste notebook é transformar as tabelas relacionais em uma visão consolidada com uma linha por `SK_ID_CURR`, compatível com a base de aplicações preparada no Notebook 03.

### 2. Objetivos

Este notebook possui os seguintes objetivos:

1. carregar e validar as tabelas auxiliares do Home Credit;
2. documentar suas chaves e granularidades;
3. avaliar a disponibilidade temporal das informações;
4. identificar códigos especiais e possíveis inconsistências;
5. criar features agregadas sem utilizar o `TARGET`;
6. produzir uma tabela consolidada com uma linha por cliente;
7. combinar as agregações com as bases de `train`, `validation` e holdout `test`;
8. garantir que o merge não altere a quantidade de clientes das amostras;
9. registrar as definições e origens das features;
10. persistir as bases consolidadas e seus metadados.

As transformações serão determinísticas e independentes do `TARGET`. Decisões que dependem da distribuição dos dados continuarão restritas à amostra de treino.

### 3. Tabelas utilizadas

As seguintes tabelas serão consideradas:

| Tabela | Granularidade principal | Conteúdo |
|---|---|---|
| `bureau.csv` | Um crédito registrado no bureau | Histórico de créditos do cliente em outras instituições |
| `bureau_balance.csv` | Um mês de um crédito do bureau | Evolução mensal dos créditos registrados no bureau |
| `previous_application.csv` | Uma aplicação anterior | Solicitações anteriores realizadas pelo cliente |
| `installments_payments.csv` | Um pagamento de parcela | Valores previstos e efetivamente pagos |
| `credit_card_balance.csv` | Um mês de contrato de cartão | Saldos, limites, pagamentos e utilização |
| `POS_CASH_balance.csv` | Um mês de contrato POS/CASH | Saldo de contratos e comportamento de atraso |

As tabelas utilizam duas chaves principais:

- `SK_ID_CURR`: identifica o cliente e sua aplicação atual;
- `SK_ID_PREV`: identifica uma aplicação ou contrato anterior.

As tabelas de bureau também utilizam:

- `SK_ID_BUREAU`: identifica um contrato registrado no bureau.

### 4. Estratégia de agregação

As tabelas auxiliares possuem múltiplos registros para um mesmo cliente. Para combiná-las com a base de aplicações, serão construídas agregações em níveis sucessivos quando necessário.

Exemplos:

- `bureau_balance` será agregado por `SK_ID_BUREAU` e posteriormente associado ao cliente por meio de `bureau`;
- pagamentos de parcelas poderão ser resumidos primeiro por contrato anterior e depois por `SK_ID_CURR`;
- saldos mensais de cartão e POS/CASH poderão ser agregados por contrato e posteriormente por cliente;
- aplicações anteriores serão resumidas diretamente por `SK_ID_CURR` ou por grupos de status.

O resultado final deverá obedecer à seguinte restrição:

$$
\text{uma linha de features históricas por } \texttt{SK\_ID\_CURR}
$$

O merge com a base principal será realizado com `left join`, preservando todos os clientes das amostras de modelagem.

### 5. Prevenção de data leakage

As features históricas somente serão válidas se representarem informações disponíveis até o momento da aplicação atual.

Por isso, serão avaliadas as variáveis temporais presentes nas tabelas auxiliares, como:

- `DAYS_CREDIT`;
- `DAYS_CREDIT_ENDDATE`;
- `DAYS_ENDDATE_FACT`;
- `DAYS_DECISION`;
- `DAYS_INSTALMENT`;
- `DAYS_ENTRY_PAYMENT`;
- `MONTHS_BALANCE`.

Valores temporais posteriores à aplicação de referência deverão ser investigados antes de serem utilizados.

O `TARGET` não será incorporado às tabelas auxiliares durante a construção das features. Ele será utilizado apenas posteriormente, nas etapas de análise e modelagem.

Também não serão calculadas agregações separadamente para bons e maus clientes, pois isso introduziria informação da variável resposta no processo de preparação.

### 6. Ausência de histórico

A ausência de registros em uma tabela auxiliar não será interpretada automaticamente como valor zero.

Ela pode representar situações diferentes:

- o cliente nunca utilizou determinado produto;
- não existe histórico disponível;
- a informação não foi reportada;
- o cliente não foi localizado naquela fonte.

Após o merge, serão criadas flags de disponibilidade para indicar se o cliente possui registros em cada fonte. As contagens poderão receber zero quando a ausência representar efetivamente nenhum registro, enquanto médias, valores e indicadores comportamentais poderão permanecer como missing.

## Configuração do Ambiente

Esta seção prepara o ambiente de execução, instala as dependências do projeto, carrega as funções compartilhadas e configura os caminhos utilizados para leitura e persistência dos dados.

Será utilizado o mesmo bootstrap dos notebooks anteriores para manter a padronização do projeto.

In [1]:
# Inicialização do projeto
import os
import sys
import subprocess
from pathlib import Path
import importlib


IN_COLAB = "google.colab" in sys.modules

REPOSITORY_URL = (
    "https://github.com/angeruzzi/"
    "home-credit-default-risk.git"
)

if IN_COLAB:
    PROJECT_ROOT = Path(
        "/content/home-credit-default-risk"
    )

    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                REPOSITORY_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    else:
        subprocess.run(
            [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
                "origin",
                "main",
            ],
            check=True,
        )

else:
    current_directory = Path.cwd().resolve()

    PROJECT_ROOT = (
        current_directory.parent
        if current_directory.name == "notebooks"
        else current_directory
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(
    f"Raiz do projeto: {PROJECT_ROOT}"
)

Raiz do projeto: /content/home-credit-default-risk


In [2]:
# Instalação das dependências
REQUIREMENTS_PATH = (
    PROJECT_ROOT / "requirements.txt"
)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(
        "Arquivo não encontrado: "
        f"{REQUIREMENTS_PATH}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    check=True,
)

print(
    "Dependências instaladas com sucesso."
)

Dependências instaladas com sucesso.


In [3]:
# Configuração dos caminhos
import utils

importlib.reload(utils)

from utils import setup_project


paths = setup_project(
    use_google_drive_cache=True,
)

print(
    "Dados brutos:",
    paths.data_raw,
)

print(
    "Dados intermediários:",
    paths.data_interim,
)

print(
    "Dados processados:",
    paths.data_processed,
)

print(
    "Metadados:",
    paths.metadata,
)

Mounted at /content/drive
Ambiente Colab: True
Modo de armazenamento: Google Drive
Raiz do projeto: /content/home-credit-default-risk
Dados: /content/drive/MyDrive/home-credit-default-risk/data
Artefatos: /content/drive/MyDrive/home-credit-default-risk/artifacts
Relatórios: /content/home-credit-default-risk/reports
Dados brutos: /content/drive/MyDrive/home-credit-default-risk/data/raw
Dados intermediários: /content/drive/MyDrive/home-credit-default-risk/data/interim
Dados processados: /content/drive/MyDrive/home-credit-default-risk/data/processed
Metadados: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata


In [4]:
# Importação das bibliotecas
import gc
import json
import platform

from datetime import datetime, timezone
from functools import reduce

import numpy as np
import pandas as pd
import pyarrow
import sklearn

from IPython.display import display

In [5]:
# Configuração de exibição
pd.set_option(
    "display.max_columns",
    200,
)

pd.set_option(
    "display.max_rows",
    100,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

In [6]:
# Registro das versões do ambiente
environment_versions = pd.DataFrame(
    [
        {
            "component": "Python",
            "version": (
                platform.python_version()
            ),
        },
        {
            "component": "NumPy",
            "version": np.__version__,
        },
        {
            "component": "Pandas",
            "version": pd.__version__,
        },
        {
            "component": "PyArrow",
            "version": pyarrow.__version__,
        },
        {
            "component": "Scikit-learn",
            "version": sklearn.__version__,
        },
    ]
)

display(environment_versions)

,component,version
0,Python,3.13.15
1,NumPy,2.1.3
2,Pandas,2.2.3
3,PyArrow,23.0.1
4,Scikit-learn,1.6.1


In [7]:
# Persistência das versões do ambiente
environment_versions_output = (
    paths.metadata
    / "notebook_04_environment_versions.csv"
)

environment_versions.to_csv(
    environment_versions_output,
    index=False,
)

print(
    "Versões registradas em:",
    environment_versions_output,
)

Versões registradas em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/notebook_04_environment_versions.csv


In [8]:
# Função para liberação de memória
def release_memory(*objects):
    """
    Remove referências globais informadas e
    solicita a coleta de memória do Python.
    """

    global_namespace = globals()

    for object_name in objects:
        if object_name in global_namespace:
            del global_namespace[object_name]

    gc.collect()

## Inventário das Tabelas Auxiliares

Antes do processamento, serão validados:

- existência dos arquivos;
- tamanho em disco;
- quantidade de colunas;
- presença das chaves obrigatórias;
- disponibilidade das variáveis mínimas necessárias para as agregações.

Nesta etapa, somente o cabeçalho de cada CSV será carregado. Isso permite validar a estrutura sem consumir a memória necessária para ler milhões de registros.

In [9]:
# Especificação das tabelas auxiliares
AUXILIARY_TABLE_SPECS = {
    "bureau": {
        "file_name": "bureau.csv",
        "required_columns": [
            "SK_ID_CURR",
            "SK_ID_BUREAU",
        ],
        "primary_grain": (
            "Um crédito registrado no bureau"
        ),
    },
    "bureau_balance": {
        "file_name": (
            "bureau_balance.csv"
        ),
        "required_columns": [
            "SK_ID_BUREAU",
            "MONTHS_BALANCE",
            "STATUS",
        ],
        "primary_grain": (
            "Um mês de um crédito do bureau"
        ),
    },
    "previous_application": {
        "file_name": (
            "previous_application.csv"
        ),
        "required_columns": [
            "SK_ID_CURR",
            "SK_ID_PREV",
        ],
        "primary_grain": (
            "Uma aplicação anterior"
        ),
    },
    "installments_payments": {
        "file_name": (
            "installments_payments.csv"
        ),
        "required_columns": [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "NUM_INSTALMENT_NUMBER",
            "DAYS_INSTALMENT",
            "DAYS_ENTRY_PAYMENT",
            "AMT_INSTALMENT",
            "AMT_PAYMENT",
        ],
        "primary_grain": (
            "Um pagamento de parcela"
        ),
    },
    "credit_card_balance": {
        "file_name": (
            "credit_card_balance.csv"
        ),
        "required_columns": [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "MONTHS_BALANCE",
        ],
        "primary_grain": (
            "Um mês de contrato de cartão"
        ),
    },
    "pos_cash_balance": {
        "file_name": (
            "POS_CASH_balance.csv"
        ),
        "required_columns": [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "MONTHS_BALANCE",
            "SK_DPD",
            "SK_DPD_DEF",
        ],
        "primary_grain": (
            "Um mês de contrato POS/CASH"
        ),
    },
}

In [10]:
# Construção do inventário estrutural
auxiliary_inventory_records = []
auxiliary_headers = {}

for table_name, specification in (
    AUXILIARY_TABLE_SPECS.items()
):
    file_path = (
        paths.data_raw
        / specification["file_name"]
    )

    if not file_path.exists():
        auxiliary_inventory_records.append(
            {
                "table": table_name,
                "file_name": (
                    specification["file_name"]
                ),
                "file_path": str(file_path),
                "file_exists": False,
                "file_size_mb": np.nan,
                "n_columns": np.nan,
                "required_columns_ok": False,
                "missing_required_columns": (
                    "<FILE_NOT_FOUND>"
                ),
                "primary_grain": (
                    specification[
                        "primary_grain"
                    ]
                ),
            }
        )

        continue

    header = pd.read_csv(
        file_path,
        nrows=0,
    )

    columns = header.columns.tolist()

    auxiliary_headers[
        table_name
    ] = columns

    missing_required_columns = sorted(
        set(
            specification[
                "required_columns"
            ]
        )
        - set(columns)
    )

    auxiliary_inventory_records.append(
        {
            "table": table_name,
            "file_name": (
                specification["file_name"]
            ),
            "file_path": str(file_path),
            "file_exists": True,
            "file_size_mb": (
                file_path.stat().st_size
                / 1024**2
            ),
            "n_columns": len(columns),
            "required_columns_ok": (
                len(
                    missing_required_columns
                )
                == 0
            ),
            "missing_required_columns": (
                ", ".join(
                    missing_required_columns
                )
            ),
            "primary_grain": (
                specification[
                    "primary_grain"
                ]
            ),
        }
    )

auxiliary_inventory = pd.DataFrame(
    auxiliary_inventory_records
)

display(
    auxiliary_inventory.style.format(
        {
            "file_size_mb": "{:,.2f}",
            "n_columns": "{:,.0f}",
        }
    )
)

,table,file_name,file_path,file_exists,file_size_mb,n_columns,required_columns_ok,missing_required_columns,primary_grain
0,bureau,bureau.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/bureau.csv,True,162.14,17,True,,Um crédito registrado no bureau
1,bureau_balance,bureau_balance.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/bureau_balance.csv,True,358.19,3,True,,Um mês de um crédito do bureau
2,previous_application,previous_application.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/previous_application.csv,True,386.21,37,True,,Uma aplicação anterior
3,installments_payments,installments_payments.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/installments_payments.csv,True,689.62,8,True,,Um pagamento de parcela
4,credit_card_balance,credit_card_balance.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/credit_card_balance.csv,True,404.91,23,True,,Um mês de contrato de cartão
5,pos_cash_balance,POS_CASH_balance.csv,/content/drive/MyDrive/home-credit-default-risk/data/raw/POS_CASH_balance.csv,True,374.51,8,True,,Um mês de contrato POS/CASH


In [11]:
# Interrupção se algum arquivo estiver ausente
missing_files = (
    auxiliary_inventory.loc[
        ~auxiliary_inventory[
            "file_exists"
        ],
        "file_name",
    ]
    .tolist()
)

if missing_files:
    raise FileNotFoundError(
        "Arquivos auxiliares ausentes: "
        f"{missing_files}"
    )

In [12]:
# Interrupção se alguma chave estiver ausente
invalid_table_schemas = (
    auxiliary_inventory.loc[
        ~auxiliary_inventory[
            "required_columns_ok"
        ],
        [
            "table",
            "missing_required_columns",
        ],
    ]
)

if not invalid_table_schemas.empty:
    display(invalid_table_schemas)

    raise ValueError(
        "Existem tabelas auxiliares "
        "sem as colunas obrigatórias."
    )

print(
    "Arquivos e colunas obrigatórias "
    "validados com sucesso."
)

Arquivos e colunas obrigatórias validados com sucesso.


In [13]:
# Exibição das colunas por tabela
for table_name, columns in (
    auxiliary_headers.items()
):
    print(
        f"\n{table_name} "
        f"({len(columns)} colunas)"
    )

    for column in columns:
        print(f"- {column}")


bureau (17 colunas)
- SK_ID_CURR
- SK_ID_BUREAU
- CREDIT_ACTIVE
- CREDIT_CURRENCY
- DAYS_CREDIT
- CREDIT_DAY_OVERDUE
- DAYS_CREDIT_ENDDATE
- DAYS_ENDDATE_FACT
- AMT_CREDIT_MAX_OVERDUE
- CNT_CREDIT_PROLONG
- AMT_CREDIT_SUM
- AMT_CREDIT_SUM_DEBT
- AMT_CREDIT_SUM_LIMIT
- AMT_CREDIT_SUM_OVERDUE
- CREDIT_TYPE
- DAYS_CREDIT_UPDATE
- AMT_ANNUITY

bureau_balance (3 colunas)
- SK_ID_BUREAU
- MONTHS_BALANCE
- STATUS

previous_application (37 colunas)
- SK_ID_PREV
- SK_ID_CURR
- NAME_CONTRACT_TYPE
- AMT_ANNUITY
- AMT_APPLICATION
- AMT_CREDIT
- AMT_DOWN_PAYMENT
- AMT_GOODS_PRICE
- WEEKDAY_APPR_PROCESS_START
- HOUR_APPR_PROCESS_START
- FLAG_LAST_APPL_PER_CONTRACT
- NFLAG_LAST_APPL_IN_DAY
- RATE_DOWN_PAYMENT
- RATE_INTEREST_PRIMARY
- RATE_INTEREST_PRIVILEGED
- NAME_CASH_LOAN_PURPOSE
- NAME_CONTRACT_STATUS
- DAYS_DECISION
- NAME_PAYMENT_TYPE
- CODE_REJECT_REASON
- NAME_TYPE_SUITE
- NAME_CLIENT_TYPE
- NAME_GOODS_CATEGORY
- NAME_PORTFOLIO
- NAME_PRODUCT_TYPE
- CHANNEL_TYPE
- SELLERPLACE_AREA
- NAME_SE

### Relações entre as tabelas

As principais relações utilizadas serão:

| Origem | Chave | Destino | Cardinalidade esperada |
|---|---|---|---|
| `application` | `SK_ID_CURR` | `bureau` | Um cliente para vários créditos |
| `bureau` | `SK_ID_BUREAU` | `bureau_balance` | Um crédito para vários meses |
| `application` | `SK_ID_CURR` | `previous_application` | Um cliente para várias aplicações anteriores |
| `previous_application` | `SK_ID_PREV` | `installments_payments` | Um contrato para vários pagamentos |
| `previous_application` | `SK_ID_PREV` | `credit_card_balance` | Um contrato para vários meses |
| `previous_application` | `SK_ID_PREV` | `POS_CASH_balance` | Um contrato para vários meses |

Embora algumas tabelas transacionais também contenham `SK_ID_CURR`, o relacionamento por `SK_ID_PREV` será validado para verificar a consistência entre contrato e cliente.

In [14]:
# Resumo do volume das tabelas auxiliares
total_auxiliary_size_mb = (
    auxiliary_inventory[
        "file_size_mb"
    ]
    .sum()
)

print(
    "Volume total dos arquivos auxiliares:",
    f"{total_auxiliary_size_mb:,.2f} MB",
)

print(
    "Volume aproximado em GB:",
    f"{total_auxiliary_size_mb / 1024:,.2f} GB",
)

Volume total dos arquivos auxiliares: 2,375.59 MB
Volume aproximado em GB: 2.32 GB


In [15]:
# Persistência do inventário estrutural
auxiliary_inventory_output = (
    paths.metadata
    / "auxiliary_tables_inventory.csv"
)

auxiliary_inventory.to_csv(
    auxiliary_inventory_output,
    index=False,
)

print(
    "Inventário salvo em:",
    auxiliary_inventory_output,
)

Inventário salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/auxiliary_tables_inventory.csv


In [16]:
# Persistência do schema das tabelas
auxiliary_schema_output = (
    paths.metadata
    / "auxiliary_tables_schema.json"
)

with open(
    auxiliary_schema_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "generated_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "tables": {
                table_name: {
                    "file_name": (
                        AUXILIARY_TABLE_SPECS[
                            table_name
                        ]["file_name"]
                    ),
                    "primary_grain": (
                        AUXILIARY_TABLE_SPECS[
                            table_name
                        ]["primary_grain"]
                    ),
                    "columns": columns,
                }
                for table_name, columns
                in auxiliary_headers.items()
            },
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Schema salvo em:",
    auxiliary_schema_output,
)

Schema salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/auxiliary_tables_schema.json


## Processamento de `bureau_balance`

A tabela `bureau_balance` registra a situação mensal de créditos existentes no bureau.

Cada linha representa um mês de um contrato identificado por `SK_ID_BUREAU`.

A coluna `STATUS` possui os seguintes códigos:

| Status | Interpretação |
|---|---|
| `0` | Sem atraso no mês |
| `1` | Atraso entre 1 e 30 dias |
| `2` | Atraso entre 31 e 60 dias |
| `3` | Atraso entre 61 e 90 dias |
| `4` | Atraso entre 91 e 120 dias |
| `5` | Atraso superior a 120 dias ou situação equivalente de alta severidade |
| `C` | Crédito encerrado |
| `X` | Status desconhecido ou não informado |

As informações mensais serão agregadas por `SK_ID_BUREAU`. Posteriormente, essas features serão associadas à tabela `bureau` e agregadas por cliente.

In [17]:
# Definição do caminho de bureau_balance
BUREAU_BALANCE_PATH = (
    paths.data_raw
    / "bureau_balance.csv"
)

BUREAU_BALANCE_COLUMNS = [
    "SK_ID_BUREAU",
    "MONTHS_BALANCE",
    "STATUS",
]

BUREAU_BALANCE_DTYPES = {
    "SK_ID_BUREAU": "int32",
    "MONTHS_BALANCE": "int16",
    "STATUS": "category",
}

In [18]:
# Carregamento otimizado de bureau_balance
bureau_balance = pd.read_csv(
    BUREAU_BALANCE_PATH,
    usecols=BUREAU_BALANCE_COLUMNS,
    dtype=BUREAU_BALANCE_DTYPES,
)

print(
    "Dimensões:",
    bureau_balance.shape,
)

print(
    "Memória utilizada:",
    f"{bureau_balance.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(bureau_balance.head())

Dimensões: (27299925, 3)
Memória utilizada: 182.25 MB


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [19]:
# Validação estrutural de bureau_balance
bureau_balance_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(bureau_balance),
        },
        {
            "validation": (
                "Contratos SK_ID_BUREAU"
            ),
            "value": (
                bureau_balance[
                    "SK_ID_BUREAU"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_BUREAU ausente"
            ),
            "value": (
                bureau_balance[
                    "SK_ID_BUREAU"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "MONTHS_BALANCE ausente"
            ),
            "value": (
                bureau_balance[
                    "MONTHS_BALANCE"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": "STATUS ausente",
            "value": (
                bureau_balance[
                    "STATUS"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "MONTHS_BALANCE positivo"
            ),
            "value": (
                bureau_balance[
                    "MONTHS_BALANCE"
                ]
                .gt(0)
                .sum()
            ),
        },
        {
            "validation": (
                "Pares contrato/mês duplicados"
            ),
            "value": (
                bureau_balance
                .duplicated(
                    subset=[
                        "SK_ID_BUREAU",
                        "MONTHS_BALANCE",
                    ]
                )
                .sum()
            ),
        },
    ]
)

display(bureau_balance_validation)

,validation,value
0,Registros,27299925
1,Contratos SK_ID_BUREAU,817395
2,SK_ID_BUREAU ausente,0
3,MONTHS_BALANCE ausente,0
4,STATUS ausente,0
5,MONTHS_BALANCE positivo,0
6,Pares contrato/mês duplicados,0


In [20]:
# Validação dos códigos de status
EXPECTED_BUREAU_BALANCE_STATUS = {
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "C",
    "X",
}

observed_bureau_balance_status = set(
    bureau_balance["STATUS"]
    .dropna()
    .astype("string")
    .unique()
    .tolist()
)

unexpected_bureau_balance_status = (
    observed_bureau_balance_status
    - EXPECTED_BUREAU_BALANCE_STATUS
)

print(
    "Status observados:",
    sorted(
        observed_bureau_balance_status
    ),
)

if unexpected_bureau_balance_status:
    raise ValueError(
        "Status inesperados em bureau_balance: "
        f"{sorted(unexpected_bureau_balance_status)}"
    )

Status observados: ['0', '1', '2', '3', '4', '5', 'C', 'X']


In [21]:
# Distribuição dos status de bureau_balance
bureau_balance_status_distribution = (
    bureau_balance["STATUS"]
    .value_counts(
        dropna=False,
    )
    .rename_axis("status")
    .reset_index(name="n_records")
)

bureau_balance_status_distribution[
    "proportion"
] = (
    bureau_balance_status_distribution[
        "n_records"
    ]
    / len(bureau_balance)
)

display(
    bureau_balance_status_distribution.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,status,n_records,proportion
0,C,"13,646,993",49.99%
1,0,"7,499,507",27.47%
2,X,"5,810,482",21.28%
3,1,"242,347",0.89%
4,5,"62,406",0.23%
5,2,"23,419",0.09%
6,3,"8,924",0.03%
7,4,"5,847",0.02%


In [22]:
# Agregação temporal por SK_ID_BUREAU
bureau_balance_temporal_agg = (
    bureau_balance
    .groupby(
        "SK_ID_BUREAU",
        observed=True,
    )
    .agg(
        BB_MONTHS_COUNT=(
            "MONTHS_BALANCE",
            "size",
        ),
        BB_MONTHS_MIN=(
            "MONTHS_BALANCE",
            "min",
        ),
        BB_MONTHS_MAX=(
            "MONTHS_BALANCE",
            "max",
        ),
    )
    .reset_index()
)

bureau_balance_temporal_agg[
    "BB_HISTORY_LENGTH_MONTHS"
] = (
    bureau_balance_temporal_agg[
        "BB_MONTHS_MAX"
    ]
    - bureau_balance_temporal_agg[
        "BB_MONTHS_MIN"
    ]
    + 1
)

bureau_balance_temporal_agg[
    "BB_HISTORY_COVERAGE_RATIO"
] = (
    bureau_balance_temporal_agg[
        "BB_MONTHS_COUNT"
    ]
    / bureau_balance_temporal_agg[
        "BB_HISTORY_LENGTH_MONTHS"
    ]
)

display(
    bureau_balance_temporal_agg.head()
)

,SK_ID_BUREAU,BB_MONTHS_COUNT,BB_MONTHS_MIN,BB_MONTHS_MAX,BB_HISTORY_LENGTH_MONTHS,BB_HISTORY_COVERAGE_RATIO
0,5001709,97,-96,0,97,1.0000
1,5001710,83,-82,0,83,1.0000
2,5001711,4,-3,0,4,1.0000
3,5001712,19,-18,0,19,1.0000
4,5001713,22,-21,0,22,1.0000


In [23]:
# Contagem dos status por SK_ID_BUREAU
bureau_balance_status_counts = (
    bureau_balance
    .groupby(
        [
            "SK_ID_BUREAU",
            "STATUS",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for status in sorted(
    EXPECTED_BUREAU_BALANCE_STATUS
):
    if status not in (
        bureau_balance_status_counts
        .columns
        .astype(str)
    ):
        bureau_balance_status_counts[
            status
        ] = 0

bureau_balance_status_counts.columns = [
    str(column)
    for column in (
        bureau_balance_status_counts.columns
    )
]

bureau_balance_status_counts = (
    bureau_balance_status_counts[
        [
            "0",
            "1",
            "2",
            "3",
            "4",
            "5",
            "C",
            "X",
        ]
    ]
    .rename(
        columns={
            status: (
                f"BB_STATUS_{status}_COUNT"
            )
            for status in [
                "0",
                "1",
                "2",
                "3",
                "4",
                "5",
                "C",
                "X",
            ]
        }
    )
    .reset_index()
)

display(
    bureau_balance_status_counts.head()
)

,SK_ID_BUREAU,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_COUNT,BB_STATUS_3_COUNT,BB_STATUS_4_COUNT,BB_STATUS_5_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT
0,5001709,0,0,0,0,0,0,86,11
1,5001710,5,0,0,0,0,0,48,30
2,5001711,3,0,0,0,0,0,0,1
3,5001712,10,0,0,0,0,0,9,0
4,5001713,0,0,0,0,0,0,0,22


In [24]:
# Combinação das agregações por contrato
bureau_balance_by_bureau = (
    bureau_balance_temporal_agg
    .merge(
        bureau_balance_status_counts,
        on="SK_ID_BUREAU",
        how="left",
        validate="one_to_one",
    )
)

In [25]:
# Criação das features de atraso
delinquency_status_columns = [
    f"BB_STATUS_{status}_COUNT"
    for status in [
        "1",
        "2",
        "3",
        "4",
        "5",
    ]
]

severe_delinquency_status_columns = [
    f"BB_STATUS_{status}_COUNT"
    for status in [
        "3",
        "4",
        "5",
    ]
]

bureau_balance_by_bureau[
    "BB_DPD_MONTHS_COUNT"
] = (
    bureau_balance_by_bureau[
        delinquency_status_columns
    ]
    .sum(axis=1)
)

bureau_balance_by_bureau[
    "BB_SEVERE_DPD_MONTHS_COUNT"
] = (
    bureau_balance_by_bureau[
        severe_delinquency_status_columns
    ]
    .sum(axis=1)
)

bureau_balance_by_bureau[
    "BB_DPD_MONTHS_RATIO"
] = (
    bureau_balance_by_bureau[
        "BB_DPD_MONTHS_COUNT"
    ]
    / bureau_balance_by_bureau[
        "BB_MONTHS_COUNT"
    ]
)

bureau_balance_by_bureau[
    "BB_SEVERE_DPD_MONTHS_RATIO"
] = (
    bureau_balance_by_bureau[
        "BB_SEVERE_DPD_MONTHS_COUNT"
    ]
    / bureau_balance_by_bureau[
        "BB_MONTHS_COUNT"
    ]
)

bureau_balance_by_bureau[
    "BB_CLOSED_MONTHS_RATIO"
] = (
    bureau_balance_by_bureau[
        "BB_STATUS_C_COUNT"
    ]
    / bureau_balance_by_bureau[
        "BB_MONTHS_COUNT"
    ]
)

bureau_balance_by_bureau[
    "BB_UNKNOWN_MONTHS_RATIO"
] = (
    bureau_balance_by_bureau[
        "BB_STATUS_X_COUNT"
    ]
    / bureau_balance_by_bureau[
        "BB_MONTHS_COUNT"
    ]
)

In [26]:
# Cálculo do pior nível de atraso
bureau_balance_by_bureau[
    "BB_WORST_DPD_LEVEL"
] = np.nan

for status_level in range(0, 6):
    status_column = (
        f"BB_STATUS_{status_level}_COUNT"
    )

    bureau_balance_by_bureau.loc[
        bureau_balance_by_bureau[
            status_column
        ].gt(0),
        "BB_WORST_DPD_LEVEL",
    ] = status_level

In [27]:
# Criação das flags de atraso
bureau_balance_by_bureau[
    "BB_HAS_DPD"
] = (
    bureau_balance_by_bureau[
        "BB_DPD_MONTHS_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

bureau_balance_by_bureau[
    "BB_HAS_SEVERE_DPD"
] = (
    bureau_balance_by_bureau[
        "BB_SEVERE_DPD_MONTHS_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

In [28]:
# Validação da agregação por contrato
n_original_bureau_ids = (
    bureau_balance[
        "SK_ID_BUREAU"
    ]
    .nunique()
)

n_aggregated_bureau_ids = (
    bureau_balance_by_bureau[
        "SK_ID_BUREAU"
    ]
    .nunique()
)

if (
    n_original_bureau_ids
    != n_aggregated_bureau_ids
):
    raise ValueError(
        "Quantidade de contratos divergente "
        "após a agregação de bureau_balance."
    )

if bureau_balance_by_bureau[
    "SK_ID_BUREAU"
].duplicated().any():
    raise ValueError(
        "A agregação possui contratos duplicados."
    )

if (
    bureau_balance_by_bureau[
        "BB_HISTORY_COVERAGE_RATIO"
    ]
    .gt(1)
    .any()
):
    raise ValueError(
        "Foram encontradas coberturas "
        "temporais superiores a 1."
    )

print(
    "Agregação validada:",
    f"{n_aggregated_bureau_ids:,}",
    "contratos.",
)

Agregação validada: 817,395 contratos.


In [29]:
# Perfil das features de bureau_balance
bureau_balance_agg_profile = (
    bureau_balance_by_bureau
    .drop(
        columns=["SK_ID_BUREAU"]
    )
    .describe(
        percentiles=[
            0.01,
            0.50,
            0.95,
            0.99,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "feature",
            "count": "n_non_missing",
            "50%": "median",
        }
    )
)

display(
    bureau_balance_agg_profile.style.format(
        {
            "n_non_missing": "{:,.0f}",
            "mean": "{:,.4f}",
            "std": "{:,.4f}",
            "min": "{:,.4f}",
            "1%": "{:,.4f}",
            "median": "{:,.4f}",
            "95%": "{:,.4f}",
            "99%": "{:,.4f}",
            "max": "{:,.4f}",
        }
    )
)

,feature,n_non_missing,mean,std,min,1%,median,95%,99%,max
0,BB_MONTHS_COUNT,"817,395",33.3987,25.7947,1.0000,2.0000,26.0000,90.0000,97.0000,97.0000
1,BB_MONTHS_MIN,"817,395",-39.4328,28.1326,-96.0000,-96.0000,-34.0000,-4.0000,-1.0000,0.0000
2,BB_MONTHS_MAX,"817,395",-7.0342,17.0269,-96.0000,-80.0000,0.0000,0.0000,0.0000,0.0000
3,BB_HISTORY_LENGTH_MONTHS,"817,395",33.3987,25.7947,1.0000,2.0000,26.0000,90.0000,97.0000,97.0000
4,BB_HISTORY_COVERAGE_RATIO,"817,395",1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
5,BB_STATUS_0_COUNT,"817,395",9.1749,10.3119,0.0000,0.0000,6.0000,29.0000,49.0000,97.0000
6,BB_STATUS_1_COUNT,"817,395",0.2965,1.2975,0.0000,0.0000,0.0000,2.0000,6.0000,56.0000
7,BB_STATUS_2_COUNT,"817,395",0.0287,0.2786,0.0000,0.0000,0.0000,0.0000,1.0000,39.0000
8,BB_STATUS_3_COUNT,"817,395",0.0109,0.1424,0.0000,0.0000,0.0000,0.0000,0.0000,18.0000
9,BB_STATUS_4_COUNT,"817,395",0.0072,0.1055,0.0000,0.0000,0.0000,0.0000,0.0000,12.0000


In [30]:
# Persistência de bureau_balance agregado
bureau_balance_by_bureau_output = (
    paths.data_interim
    / "bureau_balance_by_bureau.parquet"
)

bureau_balance_by_bureau.to_parquet(
    bureau_balance_by_bureau_output,
    index=False,
)

print(
    "Agregação salva em:",
    bureau_balance_by_bureau_output,
)

print(
    "Tamanho:",
    f"{bureau_balance_by_bureau_output.stat().st_size / 1024**2:,.2f} MB",
)

Agregação salva em: /content/drive/MyDrive/home-credit-default-risk/data/interim/bureau_balance_by_bureau.parquet
Tamanho: 11.36 MB


In [31]:
# Persistência dos metadados de bureau_balance
bureau_balance_status_distribution.to_csv(
    paths.metadata
    / "bureau_balance_status_distribution.csv",
    index=False,
)

bureau_balance_agg_profile.to_csv(
    paths.metadata
    / "bureau_balance_aggregated_profile.csv",
    index=False,
)

In [32]:
# Liberação da tabela mensal de bureau_balance
release_memory(
    "bureau_balance",
    "bureau_balance_temporal_agg",
    "bureau_balance_status_counts",
)

print(
    "Tabela mensal removida da memória."
)

Tabela mensal removida da memória.


## Processamento de `bureau`

A tabela `bureau` contém créditos do cliente reportados por outras instituições.

As agregações buscarão representar:

- quantidade de contratos;
- contratos ativos, encerrados, vendidos ou classificados como `Bad debt`;
- valores de crédito, dívida e atraso;
- idade e duração dos contratos;
- diversidade de produtos e moedas;
- histórico mensal de atraso proveniente de `bureau_balance`.

Valores positivos em `DAYS_CREDIT_ENDDATE` não representam necessariamente data leakage, pois podem corresponder ao vencimento futuro previsto de um contrato já existente. Por outro lado, `DAYS_CREDIT` positivo indicaria um crédito iniciado após a aplicação atual e deverá ser investigado.

In [33]:
# Definição do caminho e tipos de bureau
BUREAU_PATH = (
    paths.data_raw
    / "bureau.csv"
)

BUREAU_DTYPES = {
    "SK_ID_CURR": "int32",
    "SK_ID_BUREAU": "int32",
    "CREDIT_ACTIVE": "category",
    "CREDIT_CURRENCY": "category",
    "CREDIT_TYPE": "category",
    "DAYS_CREDIT": "int32",
    "CREDIT_DAY_OVERDUE": "int32",
    "DAYS_CREDIT_ENDDATE": "float32",
    "DAYS_ENDDATE_FACT": "float32",
    "AMT_CREDIT_MAX_OVERDUE": "float32",
    "CNT_CREDIT_PROLONG": "float32",
    "AMT_CREDIT_SUM": "float32",
    "AMT_CREDIT_SUM_DEBT": "float32",
    "AMT_CREDIT_SUM_LIMIT": "float32",
    "AMT_CREDIT_SUM_OVERDUE": "float32",
    "DAYS_CREDIT_UPDATE": "int32",
    "AMT_ANNUITY": "float32",
}

In [34]:
# Carregamento otimizado de bureau
bureau = pd.read_csv(
    BUREAU_PATH,
    dtype=BUREAU_DTYPES,
)

print(
    "Dimensões:",
    bureau.shape,
)

print(
    "Memória utilizada:",
    f"{bureau.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(bureau.head())

Dimensões: (1716428, 17)
Memória utilizada: 96.58 MB


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0000,-153.0000,NaN,0.0000,"91,323.0000",0.0000,NaN,0.0000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,"1,075.0000",NaN,NaN,0.0000,"225,000.0000","171,342.0000",NaN,0.0000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0000,NaN,NaN,0.0000,"464,323.5000",NaN,NaN,0.0000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0.0000,"90,000.0000",NaN,NaN,0.0000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,"1,197.0000",NaN,"77,674.5000",0.0000,"2,700,000.0000",NaN,NaN,0.0000,Consumer credit,-21,NaN


In [35]:
# Validação das chaves de bureau
bureau_key_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(bureau),
        },
        {
            "validation": (
                "Clientes SK_ID_CURR"
            ),
            "value": (
                bureau[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "Contratos SK_ID_BUREAU"
            ),
            "value": (
                bureau[
                    "SK_ID_BUREAU"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_CURR ausente"
            ),
            "value": (
                bureau[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_BUREAU ausente"
            ),
            "value": (
                bureau[
                    "SK_ID_BUREAU"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_BUREAU duplicado"
            ),
            "value": (
                bureau[
                    "SK_ID_BUREAU"
                ]
                .duplicated()
                .sum()
            ),
        },
        {
            "validation": (
                "DAYS_CREDIT positivo"
            ),
            "value": (
                bureau[
                    "DAYS_CREDIT"
                ]
                .gt(0)
                .sum()
            ),
        },
    ]
)

display(bureau_key_validation)

,validation,value
0,Registros,1716428
1,Clientes SK_ID_CURR,305811
2,Contratos SK_ID_BUREAU,1716428
3,SK_ID_CURR ausente,0
4,SK_ID_BUREAU ausente,0
5,SK_ID_BUREAU duplicado,0
6,DAYS_CREDIT positivo,0


In [36]:
# Interrupção se a chave de bureau for inválida
if bureau[
    "SK_ID_BUREAU"
].isna().any():
    raise ValueError(
        "SK_ID_BUREAU possui valores ausentes."
    )

if bureau[
    "SK_ID_BUREAU"
].duplicated().any():
    raise ValueError(
        "SK_ID_BUREAU não é único em bureau."
    )

if bureau[
    "DAYS_CREDIT"
].gt(0).any():
    raise ValueError(
        "Existem créditos com DAYS_CREDIT "
        "posterior à aplicação atual."
    )

In [37]:
# Distribuição de CREDIT_ACTIVE
bureau_credit_active_distribution = (
    bureau["CREDIT_ACTIVE"]
    .value_counts(
        dropna=False,
    )
    .rename_axis("credit_active")
    .reset_index(name="n_records")
)

bureau_credit_active_distribution[
    "proportion"
] = (
    bureau_credit_active_distribution[
        "n_records"
    ]
    / len(bureau)
)

display(
    bureau_credit_active_distribution.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,credit_active,n_records,proportion
0,Closed,"1,079,273",62.88%
1,Active,"630,607",36.74%
2,Sold,"6,527",0.38%
3,Bad debt,21,0.00%


In [38]:
# Validação da cobertura de bureau_balance
bureau_ids = set(
    bureau["SK_ID_BUREAU"]
)

bureau_balance_ids = set(
    bureau_balance_by_bureau[
        "SK_ID_BUREAU"
    ]
)

bureau_balance_without_bureau = (
    bureau_balance_ids
    - bureau_ids
)

bureau_without_balance = (
    bureau_ids
    - bureau_balance_ids
)

print(
    "Contratos em bureau com histórico mensal:",
    f"{len(bureau_ids & bureau_balance_ids):,}",
)

print(
    "Contratos de bureau sem histórico mensal:",
    f"{len(bureau_without_balance):,}",
)

print(
    "Contratos mensais sem correspondência em bureau:",
    f"{len(bureau_balance_without_bureau):,}",
)

Contratos em bureau com histórico mensal: 774,354
Contratos de bureau sem histórico mensal: 942,074
Contratos mensais sem correspondência em bureau: 43,041


In [39]:
# Merge entre bureau e bureau_balance agregado
bureau_enriched = (
    bureau
    .merge(
        bureau_balance_by_bureau,
        on="SK_ID_BUREAU",
        how="left",
        validate="one_to_one",
    )
)

if len(bureau_enriched) != len(bureau):
    raise ValueError(
        "O merge alterou a quantidade "
        "de contratos de bureau."
    )

print(
    "Dimensões após o merge:",
    bureau_enriched.shape,
)

Dimensões após o merge: (1716428, 39)


In [40]:
# Função de divisão segura
def safe_divide(
    numerator,
    denominator,
):
    numerator = pd.to_numeric(
        numerator,
        errors="coerce",
    )

    denominator = pd.to_numeric(
        denominator,
        errors="coerce",
    )

    valid_denominator = (
        denominator.notna()
        & np.isfinite(denominator)
        & denominator.gt(0)
    )

    result = pd.Series(
        np.nan,
        index=numerator.index,
        dtype="float64",
    )

    result.loc[valid_denominator] = (
        numerator.loc[valid_denominator]
        / denominator.loc[valid_denominator]
    )

    return result

In [41]:
# Criação das features por contrato de bureau
bureau_enriched[
    "BU_CREDIT_AGE_YEARS"
] = (
    -bureau_enriched["DAYS_CREDIT"]
    / 365.25
)

bureau_enriched[
    "BU_PLANNED_DURATION_YEARS"
] = (
    (
        bureau_enriched[
            "DAYS_CREDIT_ENDDATE"
        ]
        - bureau_enriched[
            "DAYS_CREDIT"
        ]
    )
    / 365.25
)

bureau_enriched[
    "BU_DEBT_CREDIT_RATIO"
] = safe_divide(
    bureau_enriched[
        "AMT_CREDIT_SUM_DEBT"
    ],
    bureau_enriched[
        "AMT_CREDIT_SUM"
    ],
)

bureau_enriched[
    "BU_OVERDUE_CREDIT_RATIO"
] = safe_divide(
    bureau_enriched[
        "AMT_CREDIT_SUM_OVERDUE"
    ],
    bureau_enriched[
        "AMT_CREDIT_SUM"
    ],
)

In [42]:
# Criação das flags por contrato
bureau_enriched[
    "BU_IS_ACTIVE"
] = (
    bureau_enriched[
        "CREDIT_ACTIVE"
    ]
    .astype("string")
    .eq("Active")
    .astype("int8")
)

bureau_enriched[
    "BU_IS_CLOSED"
] = (
    bureau_enriched[
        "CREDIT_ACTIVE"
    ]
    .astype("string")
    .eq("Closed")
    .astype("int8")
)

bureau_enriched[
    "BU_IS_SOLD"
] = (
    bureau_enriched[
        "CREDIT_ACTIVE"
    ]
    .astype("string")
    .eq("Sold")
    .astype("int8")
)

bureau_enriched[
    "BU_IS_BAD_DEBT"
] = (
    bureau_enriched[
        "CREDIT_ACTIVE"
    ]
    .astype("string")
    .eq("Bad debt")
    .astype("int8")
)

bureau_enriched[
    "BU_HAS_CURRENT_DEBT"
] = (
    bureau_enriched[
        "AMT_CREDIT_SUM_DEBT"
    ]
    .fillna(0)
    .gt(0)
    .astype("int8")
)

bureau_enriched[
    "BU_HAS_OVERDUE"
] = (
    (
        bureau_enriched[
            "CREDIT_DAY_OVERDUE"
        ]
        .fillna(0)
        .gt(0)
    )
    |
    (
        bureau_enriched[
            "AMT_CREDIT_SUM_OVERDUE"
        ]
        .fillna(0)
        .gt(0)
    )
).astype("int8")

bureau_enriched[
    "BU_HAS_BALANCE_HISTORY"
] = (
    bureau_enriched[
        "BB_MONTHS_COUNT"
    ]
    .notna()
    .astype("int8")
)

In [43]:
# Criação dos valores condicionais a contratos ativos
active_contract_mask = (
    bureau_enriched[
        "BU_IS_ACTIVE"
    ]
    .eq(1)
)

bureau_enriched[
    "BU_ACTIVE_CREDIT_AMOUNT"
] = (
    bureau_enriched[
        "AMT_CREDIT_SUM"
    ]
    .where(active_contract_mask)
)

bureau_enriched[
    "BU_ACTIVE_DEBT_AMOUNT"
] = (
    bureau_enriched[
        "AMT_CREDIT_SUM_DEBT"
    ]
    .where(active_contract_mask)
)

bureau_enriched[
    "BU_ACTIVE_OVERDUE_AMOUNT"
] = (
    bureau_enriched[
        "AMT_CREDIT_SUM_OVERDUE"
    ]
    .where(active_contract_mask)
)

In [44]:
# Agregação de bureau por SK_ID_CURR
bureau_by_client = (
    bureau_enriched
    .groupby(
        "SK_ID_CURR",
        observed=True,
    )
    .agg(
        BU_CONTRACT_COUNT=(
            "SK_ID_BUREAU",
            "nunique",
        ),
        BU_CREDIT_TYPE_COUNT=(
            "CREDIT_TYPE",
            "nunique",
        ),
        BU_CURRENCY_COUNT=(
            "CREDIT_CURRENCY",
            "nunique",
        ),
        BU_ACTIVE_COUNT=(
            "BU_IS_ACTIVE",
            "sum",
        ),
        BU_CLOSED_COUNT=(
            "BU_IS_CLOSED",
            "sum",
        ),
        BU_SOLD_COUNT=(
            "BU_IS_SOLD",
            "sum",
        ),
        BU_BAD_DEBT_COUNT=(
            "BU_IS_BAD_DEBT",
            "sum",
        ),
        BU_CURRENT_DEBT_COUNT=(
            "BU_HAS_CURRENT_DEBT",
            "sum",
        ),
        BU_OVERDUE_CONTRACT_COUNT=(
            "BU_HAS_OVERDUE",
            "sum",
        ),
        BU_BALANCE_HISTORY_COUNT=(
            "BU_HAS_BALANCE_HISTORY",
            "sum",
        ),
        BU_CREDIT_AGE_YEARS_MIN=(
            "BU_CREDIT_AGE_YEARS",
            "min",
        ),
        BU_CREDIT_AGE_YEARS_MEAN=(
            "BU_CREDIT_AGE_YEARS",
            "mean",
        ),
        BU_CREDIT_AGE_YEARS_MAX=(
            "BU_CREDIT_AGE_YEARS",
            "max",
        ),
        BU_PLANNED_DURATION_MEAN=(
            "BU_PLANNED_DURATION_YEARS",
            "mean",
        ),
        BU_PLANNED_DURATION_MAX=(
            "BU_PLANNED_DURATION_YEARS",
            "max",
        ),
        BU_CREDIT_AMOUNT_SUM=(
            "AMT_CREDIT_SUM",
            "sum",
        ),
        BU_CREDIT_AMOUNT_MEAN=(
            "AMT_CREDIT_SUM",
            "mean",
        ),
        BU_CREDIT_AMOUNT_MAX=(
            "AMT_CREDIT_SUM",
            "max",
        ),
        BU_DEBT_AMOUNT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum",
        ),
        BU_DEBT_AMOUNT_MEAN=(
            "AMT_CREDIT_SUM_DEBT",
            "mean",
        ),
        BU_DEBT_AMOUNT_MAX=(
            "AMT_CREDIT_SUM_DEBT",
            "max",
        ),
        BU_OVERDUE_AMOUNT_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum",
        ),
        BU_OVERDUE_AMOUNT_MAX=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max",
        ),
        BU_MAX_OVERDUE_MEAN=(
            "AMT_CREDIT_MAX_OVERDUE",
            "mean",
        ),
        BU_MAX_OVERDUE_MAX=(
            "AMT_CREDIT_MAX_OVERDUE",
            "max",
        ),
        BU_ANNUITY_SUM=(
            "AMT_ANNUITY",
            "sum",
        ),
        BU_ANNUITY_MEAN=(
            "AMT_ANNUITY",
            "mean",
        ),
        BU_PROLONG_COUNT_SUM=(
            "CNT_CREDIT_PROLONG",
            "sum",
        ),
        BU_DEBT_CREDIT_RATIO_MEAN=(
            "BU_DEBT_CREDIT_RATIO",
            "mean",
        ),
        BU_DEBT_CREDIT_RATIO_MAX=(
            "BU_DEBT_CREDIT_RATIO",
            "max",
        ),
        BU_OVERDUE_CREDIT_RATIO_MEAN=(
            "BU_OVERDUE_CREDIT_RATIO",
            "mean",
        ),
        BU_OVERDUE_CREDIT_RATIO_MAX=(
            "BU_OVERDUE_CREDIT_RATIO",
            "max",
        ),
        BU_ACTIVE_CREDIT_AMOUNT_SUM=(
            "BU_ACTIVE_CREDIT_AMOUNT",
            "sum",
        ),
        BU_ACTIVE_DEBT_AMOUNT_SUM=(
            "BU_ACTIVE_DEBT_AMOUNT",
            "sum",
        ),
        BU_ACTIVE_OVERDUE_AMOUNT_SUM=(
            "BU_ACTIVE_OVERDUE_AMOUNT",
            "sum",
        ),
        BU_BB_MONTHS_COUNT_SUM=(
            "BB_MONTHS_COUNT",
            "sum",
        ),
        BU_BB_MONTHS_COUNT_MAX=(
            "BB_MONTHS_COUNT",
            "max",
        ),
        BU_BB_DPD_MONTHS_COUNT_SUM=(
            "BB_DPD_MONTHS_COUNT",
            "sum",
        ),
        BU_BB_SEVERE_DPD_COUNT_SUM=(
            "BB_SEVERE_DPD_MONTHS_COUNT",
            "sum",
        ),
        BU_BB_DPD_MONTHS_RATIO_MEAN=(
            "BB_DPD_MONTHS_RATIO",
            "mean",
        ),
        BU_BB_DPD_MONTHS_RATIO_MAX=(
            "BB_DPD_MONTHS_RATIO",
            "max",
        ),
        BU_BB_SEVERE_DPD_RATIO_MEAN=(
            "BB_SEVERE_DPD_MONTHS_RATIO",
            "mean",
        ),
        BU_BB_SEVERE_DPD_RATIO_MAX=(
            "BB_SEVERE_DPD_MONTHS_RATIO",
            "max",
        ),
        BU_BB_WORST_DPD_LEVEL=(
            "BB_WORST_DPD_LEVEL",
            "max",
        ),
        BU_BB_HAS_DPD_COUNT=(
            "BB_HAS_DPD",
            "sum",
        ),
        BU_BB_HAS_SEVERE_DPD_COUNT=(
            "BB_HAS_SEVERE_DPD",
            "sum",
        ),
    )
    .reset_index()
)

In [45]:
# Criação das razões agregadas de bureau
bureau_by_client[
    "BU_ACTIVE_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_ACTIVE_COUNT"
    ],
    bureau_by_client[
        "BU_CONTRACT_COUNT"
    ],
)

bureau_by_client[
    "BU_CLOSED_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_CLOSED_COUNT"
    ],
    bureau_by_client[
        "BU_CONTRACT_COUNT"
    ],
)

bureau_by_client[
    "BU_OVERDUE_CONTRACT_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_OVERDUE_CONTRACT_COUNT"
    ],
    bureau_by_client[
        "BU_CONTRACT_COUNT"
    ],
)

bureau_by_client[
    "BU_BALANCE_HISTORY_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_BALANCE_HISTORY_COUNT"
    ],
    bureau_by_client[
        "BU_CONTRACT_COUNT"
    ],
)

bureau_by_client[
    "BU_TOTAL_DEBT_CREDIT_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_DEBT_AMOUNT_SUM"
    ],
    bureau_by_client[
        "BU_CREDIT_AMOUNT_SUM"
    ],
)

bureau_by_client[
    "BU_ACTIVE_DEBT_CREDIT_RATIO"
] = safe_divide(
    bureau_by_client[
        "BU_ACTIVE_DEBT_AMOUNT_SUM"
    ],
    bureau_by_client[
        "BU_ACTIVE_CREDIT_AMOUNT_SUM"
    ],
)

In [46]:
# Validação da agregação de bureau por cliente
n_bureau_clients = (
    bureau[
        "SK_ID_CURR"
    ]
    .nunique()
)

n_aggregated_clients = (
    bureau_by_client[
        "SK_ID_CURR"
    ]
    .nunique()
)

if n_bureau_clients != n_aggregated_clients:
    raise ValueError(
        "Quantidade de clientes divergente "
        "após a agregação de bureau."
    )

if bureau_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "bureau_by_client possui "
        "clientes duplicados."
    )

print(
    "Agregação validada:",
    f"{n_aggregated_clients:,}",
    "clientes.",
)

print(
    "Quantidade de features:",
    bureau_by_client.shape[1] - 1,
)

Agregação validada: 305,811 clientes.
Quantidade de features: 52


In [47]:
# Persistência das features de bureau
bureau_by_client_output = (
    paths.data_interim
    / "bureau_by_client.parquet"
)

bureau_by_client.to_parquet(
    bureau_by_client_output,
    index=False,
)

print(
    "Features salvas em:",
    bureau_by_client_output,
)

print(
    "Tamanho:",
    f"{bureau_by_client_output.stat().st_size / 1024**2:,.2f} MB",
)

Features salvas em: /content/drive/MyDrive/home-credit-default-risk/data/interim/bureau_by_client.parquet
Tamanho: 28.04 MB


In [48]:
# Persistência da distribuição de CREDIT_ACTIVE
bureau_credit_active_distribution.to_csv(
    paths.metadata
    / "bureau_credit_active_distribution.csv",
    index=False,
)

In [49]:
# Liberação das tabelas de bureau
release_memory(
    "bureau",
    "bureau_enriched",
    "bureau_balance_by_bureau",
)

print(
    "Tabelas detalhadas de bureau "
    "removidas da memória."
)

Tabelas detalhadas de bureau removidas da memória.


In [50]:
# Liberação opcional da agregação por cliente
release_memory(
    "bureau_by_client",
)

## Processamento de `previous_application`

A tabela `previous_application` contém solicitações de crédito realizadas anteriormente pelo cliente na Home Credit.

As agregações buscarão representar:

- quantidade de aplicações anteriores;
- proporções de aplicações aprovadas, recusadas e canceladas;
- valores solicitados e concedidos;
- diferença entre valor solicitado e valor aprovado;
- anuidades e pagamentos iniciais;
- quantidade prevista de pagamentos;
- recência das decisões;
- diversidade de produtos e canais;
- características específicas das aplicações aprovadas.

Algumas colunas temporais utilizam o código especial `365243` para representar informação indisponível ou não aplicável. Esses valores serão convertidos em missing antes da criação das features.

In [51]:
# Definição do caminho de previous_application
PREVIOUS_APPLICATION_PATH = (
    paths.data_raw
    / "previous_application.csv"
)

PREVIOUS_APPLICATION_COLUMNS = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "NAME_CONTRACT_TYPE",
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "RATE_DOWN_PAYMENT",
    "NAME_CONTRACT_STATUS",
    "DAYS_DECISION",
    "NAME_CLIENT_TYPE",
    "NAME_PORTFOLIO",
    "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE",
    "CNT_PAYMENT",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL",
]

PREVIOUS_APPLICATION_DTYPES = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "NAME_CONTRACT_TYPE": "category",
    "AMT_ANNUITY": "float32",
    "AMT_APPLICATION": "float32",
    "AMT_CREDIT": "float32",
    "AMT_DOWN_PAYMENT": "float32",
    "AMT_GOODS_PRICE": "float32",
    "RATE_DOWN_PAYMENT": "float32",
    "NAME_CONTRACT_STATUS": "category",
    "DAYS_DECISION": "int32",
    "NAME_CLIENT_TYPE": "category",
    "NAME_PORTFOLIO": "category",
    "NAME_PRODUCT_TYPE": "category",
    "CHANNEL_TYPE": "category",
    "CNT_PAYMENT": "float32",
    "NAME_YIELD_GROUP": "category",
    "PRODUCT_COMBINATION": "category",
    "DAYS_FIRST_DRAWING": "float32",
    "DAYS_FIRST_DUE": "float32",
    "DAYS_LAST_DUE_1ST_VERSION": "float32",
    "DAYS_LAST_DUE": "float32",
    "DAYS_TERMINATION": "float32",
    "NFLAG_INSURED_ON_APPROVAL": "float32",
}

In [52]:
# Carregamento de previous_application
previous_application = pd.read_csv(
    PREVIOUS_APPLICATION_PATH,
    usecols=PREVIOUS_APPLICATION_COLUMNS,
    dtype=PREVIOUS_APPLICATION_DTYPES,
)

print(
    "Dimensões:",
    previous_application.shape,
)

print(
    "Memória utilizada:",
    f"{previous_application.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(
    previous_application.head()
)

Dimensões: (1670214, 24)
Memória utilizada: 114.69 MB


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,RATE_DOWN_PAYMENT,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_CLIENT_TYPE,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,"1,730.4301","17,145.0000","17,145.0000",0.0000,"17,145.0000",0.0000,Approved,-73,Repeater,POS,XNA,Country-wide,12.0000,middle,POS mobile with interest,"365,243.0000",-42.0000,300.0000,-42.0000,-37.0000,0.0000
1,2802425,108129,Cash loans,"25,188.6152","607,500.0000","679,671.0000",NaN,"607,500.0000",NaN,Approved,-164,Repeater,Cash,x-sell,Contact center,36.0000,low_action,Cash X-Sell: low,"365,243.0000",-134.0000,916.0000,"365,243.0000","365,243.0000",1.0000
2,2523466,122040,Cash loans,"15,060.7354","112,500.0000","136,444.5000",NaN,"112,500.0000",NaN,Approved,-301,Repeater,Cash,x-sell,Credit and cash offices,12.0000,high,Cash X-Sell: high,"365,243.0000",-271.0000,59.0000,"365,243.0000","365,243.0000",1.0000
3,2819243,176158,Cash loans,"47,041.3359","450,000.0000","470,790.0000",NaN,"450,000.0000",NaN,Approved,-512,Repeater,Cash,x-sell,Credit and cash offices,12.0000,middle,Cash X-Sell: middle,"365,243.0000",-482.0000,-152.0000,-182.0000,-177.0000,1.0000
4,1784265,202054,Cash loans,"31,924.3945","337,500.0000","404,055.0000",NaN,"337,500.0000",NaN,Refused,-781,Repeater,Cash,walk-in,Credit and cash offices,24.0000,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
# Validação estrutural de previous_application
previous_application_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(
                previous_application
            ),
        },
        {
            "validation": (
                "Clientes SK_ID_CURR"
            ),
            "value": (
                previous_application[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "Aplicações SK_ID_PREV"
            ),
            "value": (
                previous_application[
                    "SK_ID_PREV"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_CURR ausente"
            ),
            "value": (
                previous_application[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_PREV ausente"
            ),
            "value": (
                previous_application[
                    "SK_ID_PREV"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_PREV duplicado"
            ),
            "value": (
                previous_application[
                    "SK_ID_PREV"
                ]
                .duplicated()
                .sum()
            ),
        },
        {
            "validation": (
                "DAYS_DECISION positivo"
            ),
            "value": (
                previous_application[
                    "DAYS_DECISION"
                ]
                .gt(0)
                .sum()
            ),
        },
    ]
)

display(
    previous_application_validation
)

,validation,value
0,Registros,1670214
1,Clientes SK_ID_CURR,338857
2,Aplicações SK_ID_PREV,1670214
3,SK_ID_CURR ausente,0
4,SK_ID_PREV ausente,0
5,SK_ID_PREV duplicado,0
6,DAYS_DECISION positivo,0


In [54]:
# Interrupção se a estrutura for inválida
if previous_application[
    "SK_ID_PREV"
].isna().any():
    raise ValueError(
        "SK_ID_PREV possui valores ausentes."
    )

if previous_application[
    "SK_ID_PREV"
].duplicated().any():
    raise ValueError(
        "SK_ID_PREV não é único em "
        "previous_application."
    )

if previous_application[
    "DAYS_DECISION"
].gt(0).any():
    raise ValueError(
        "Existem aplicações anteriores com "
        "DAYS_DECISION posterior à aplicação atual."
    )

In [55]:
# Distribuição de NAME_CONTRACT_STATUS
previous_status_distribution = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .value_counts(
        dropna=False,
    )
    .rename_axis("contract_status")
    .reset_index(name="n_records")
)

previous_status_distribution[
    "proportion"
] = (
    previous_status_distribution[
        "n_records"
    ]
    / len(previous_application)
)

display(
    previous_status_distribution.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,contract_status,n_records,proportion
0,Approved,"1,036,781",62.07%
1,Canceled,"316,319",18.94%
2,Refused,"290,678",17.40%
3,Unused offer,"26,436",1.58%


In [56]:
# Identificação dos códigos temporais especiais
SPECIAL_PREVIOUS_DAYS_VALUE = 365243

previous_date_columns = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

previous_special_values = pd.DataFrame(
    [
        {
            "feature": column,
            "n_special": (
                previous_application[
                    column
                ]
                .eq(
                    SPECIAL_PREVIOUS_DAYS_VALUE
                )
                .sum()
            ),
            "special_rate": (
                previous_application[
                    column
                ]
                .eq(
                    SPECIAL_PREVIOUS_DAYS_VALUE
                )
                .mean()
            ),
        }
        for column in previous_date_columns
    ]
)

display(
    previous_special_values.style.format(
        {
            "n_special": "{:,.0f}",
            "special_rate": "{:.2%}",
        }
    )
)

,feature,n_special,special_rate
0,DAYS_FIRST_DRAWING,"934,444",55.95%
1,DAYS_FIRST_DUE,"40,645",2.43%
2,DAYS_LAST_DUE_1ST_VERSION,"93,864",5.62%
3,DAYS_LAST_DUE,"211,221",12.65%
4,DAYS_TERMINATION,"225,913",13.53%


In [57]:
# Substituição dos códigos especiais por missing
for column in previous_date_columns:
    previous_application.loc[
        previous_application[
            column
        ].eq(
            SPECIAL_PREVIOUS_DAYS_VALUE
        ),
        column,
    ] = np.nan

In [58]:
# Criação das flags de status
contract_status = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .astype("string")
)

previous_application[
    "PREV_IS_APPROVED"
] = (
    contract_status
    .eq("Approved")
    .astype("int8")
)

previous_application[
    "PREV_IS_REFUSED"
] = (
    contract_status
    .eq("Refused")
    .astype("int8")
)

previous_application[
    "PREV_IS_CANCELED"
] = (
    contract_status
    .eq("Canceled")
    .astype("int8")
)

previous_application[
    "PREV_IS_UNUSED"
] = (
    contract_status
    .eq("Unused offer")
    .astype("int8")
)

In [59]:
# Criação das flags de tipo de contrato
contract_type = (
    previous_application[
        "NAME_CONTRACT_TYPE"
    ]
    .astype("string")
)

previous_application[
    "PREV_IS_CASH_LOAN"
] = (
    contract_type
    .eq("Cash loans")
    .astype("int8")
)

previous_application[
    "PREV_IS_CONSUMER_LOAN"
] = (
    contract_type
    .eq("Consumer loans")
    .astype("int8")
)

previous_application[
    "PREV_IS_REVOLVING_LOAN"
] = (
    contract_type
    .eq("Revolving loans")
    .astype("int8")
)

In [60]:
# Criação das razões financeiras
previous_application[
    "PREV_CREDIT_APPLICATION_RATIO"
] = safe_divide(
    previous_application[
        "AMT_CREDIT"
    ],
    previous_application[
        "AMT_APPLICATION"
    ],
)

previous_application[
    "PREV_DOWN_PAYMENT_GOODS_RATIO"
] = safe_divide(
    previous_application[
        "AMT_DOWN_PAYMENT"
    ],
    previous_application[
        "AMT_GOODS_PRICE"
    ],
)

previous_application[
    "PREV_ANNUITY_CREDIT_RATIO"
] = safe_divide(
    previous_application[
        "AMT_ANNUITY"
    ],
    previous_application[
        "AMT_CREDIT"
    ],
)

previous_application[
    "PREV_CREDIT_APPLICATION_DIFF"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    - previous_application[
        "AMT_APPLICATION"
    ]
)

previous_application[
    "PREV_DECISION_AGE_YEARS"
] = (
    -previous_application[
        "DAYS_DECISION"
    ]
    / 365.25
)

In [61]:
# Criação das durações planejada e observada
previous_application[
    "PREV_PLANNED_DURATION_DAYS"
] = (
    previous_application[
        "DAYS_LAST_DUE_1ST_VERSION"
    ]
    - previous_application[
        "DAYS_FIRST_DUE"
    ]
)

previous_application[
    "PREV_OBSERVED_DURATION_DAYS"
] = (
    previous_application[
        "DAYS_TERMINATION"
    ]
    - previous_application[
        "DAYS_FIRST_DUE"
    ]
)

In [62]:
# Criação das medidas condicionais a aprovações
approved_mask = (
    previous_application[
        "PREV_IS_APPROVED"
    ]
    .eq(1)
)

previous_application[
    "PREV_APPROVED_CREDIT"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    .where(approved_mask)
)

previous_application[
    "PREV_APPROVED_ANNUITY"
] = (
    previous_application[
        "AMT_ANNUITY"
    ]
    .where(approved_mask)
)

previous_application[
    "PREV_APPROVED_CNT_PAYMENT"
] = (
    previous_application[
        "CNT_PAYMENT"
    ]
    .where(approved_mask)
)

previous_application[
    "PREV_APPROVED_DAYS_DECISION"
] = (
    previous_application[
        "DAYS_DECISION"
    ]
    .where(approved_mask)
)

In [63]:
# Agregação de previous_application por cliente
previous_by_client = (
    previous_application
    .groupby(
        "SK_ID_CURR",
        observed=True,
    )
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "nunique",
        ),
        PREV_APPROVED_COUNT=(
            "PREV_IS_APPROVED",
            "sum",
        ),
        PREV_REFUSED_COUNT=(
            "PREV_IS_REFUSED",
            "sum",
        ),
        PREV_CANCELED_COUNT=(
            "PREV_IS_CANCELED",
            "sum",
        ),
        PREV_UNUSED_COUNT=(
            "PREV_IS_UNUSED",
            "sum",
        ),
        PREV_CASH_LOAN_COUNT=(
            "PREV_IS_CASH_LOAN",
            "sum",
        ),
        PREV_CONSUMER_LOAN_COUNT=(
            "PREV_IS_CONSUMER_LOAN",
            "sum",
        ),
        PREV_REVOLVING_LOAN_COUNT=(
            "PREV_IS_REVOLVING_LOAN",
            "sum",
        ),
        PREV_CONTRACT_TYPE_COUNT=(
            "NAME_CONTRACT_TYPE",
            "nunique",
        ),
        PREV_PORTFOLIO_COUNT=(
            "NAME_PORTFOLIO",
            "nunique",
        ),
        PREV_PRODUCT_TYPE_COUNT=(
            "NAME_PRODUCT_TYPE",
            "nunique",
        ),
        PREV_CHANNEL_COUNT=(
            "CHANNEL_TYPE",
            "nunique",
        ),
        PREV_PRODUCT_COMBINATION_COUNT=(
            "PRODUCT_COMBINATION",
            "nunique",
        ),
        PREV_DECISION_AGE_MIN=(
            "PREV_DECISION_AGE_YEARS",
            "min",
        ),
        PREV_DECISION_AGE_MEAN=(
            "PREV_DECISION_AGE_YEARS",
            "mean",
        ),
        PREV_DECISION_AGE_MAX=(
            "PREV_DECISION_AGE_YEARS",
            "max",
        ),
        PREV_APPLICATION_AMOUNT_SUM=(
            "AMT_APPLICATION",
            "sum",
        ),
        PREV_APPLICATION_AMOUNT_MEAN=(
            "AMT_APPLICATION",
            "mean",
        ),
        PREV_CREDIT_AMOUNT_SUM=(
            "AMT_CREDIT",
            "sum",
        ),
        PREV_CREDIT_AMOUNT_MEAN=(
            "AMT_CREDIT",
            "mean",
        ),
        PREV_CREDIT_AMOUNT_MAX=(
            "AMT_CREDIT",
            "max",
        ),
        PREV_ANNUITY_MEAN=(
            "AMT_ANNUITY",
            "mean",
        ),
        PREV_ANNUITY_MAX=(
            "AMT_ANNUITY",
            "max",
        ),
        PREV_DOWN_PAYMENT_SUM=(
            "AMT_DOWN_PAYMENT",
            "sum",
        ),
        PREV_DOWN_PAYMENT_MEAN=(
            "AMT_DOWN_PAYMENT",
            "mean",
        ),
        PREV_CNT_PAYMENT_MEAN=(
            "CNT_PAYMENT",
            "mean",
        ),
        PREV_CNT_PAYMENT_MAX=(
            "CNT_PAYMENT",
            "max",
        ),
        PREV_CREDIT_APPLICATION_RATIO_MEAN=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "mean",
        ),
        PREV_CREDIT_APPLICATION_RATIO_MAX=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "max",
        ),
        PREV_DOWN_PAYMENT_RATIO_MEAN=(
            "PREV_DOWN_PAYMENT_GOODS_RATIO",
            "mean",
        ),
        PREV_ANNUITY_CREDIT_RATIO_MEAN=(
            "PREV_ANNUITY_CREDIT_RATIO",
            "mean",
        ),
        PREV_CREDIT_APPLICATION_DIFF_MEAN=(
            "PREV_CREDIT_APPLICATION_DIFF",
            "mean",
        ),
        PREV_PLANNED_DURATION_MEAN=(
            "PREV_PLANNED_DURATION_DAYS",
            "mean",
        ),
        PREV_PLANNED_DURATION_MAX=(
            "PREV_PLANNED_DURATION_DAYS",
            "max",
        ),
        PREV_OBSERVED_DURATION_MEAN=(
            "PREV_OBSERVED_DURATION_DAYS",
            "mean",
        ),
        PREV_APPROVED_CREDIT_SUM=(
            "PREV_APPROVED_CREDIT",
            "sum",
        ),
        PREV_APPROVED_CREDIT_MEAN=(
            "PREV_APPROVED_CREDIT",
            "mean",
        ),
        PREV_APPROVED_ANNUITY_MEAN=(
            "PREV_APPROVED_ANNUITY",
            "mean",
        ),
        PREV_APPROVED_CNT_PAYMENT_MEAN=(
            "PREV_APPROVED_CNT_PAYMENT",
            "mean",
        ),
        PREV_MOST_RECENT_APPROVED_DECISION=(
            "PREV_APPROVED_DAYS_DECISION",
            "max",
        ),
        PREV_INSURED_APPROVAL_MEAN=(
            "NFLAG_INSURED_ON_APPROVAL",
            "mean",
        ),
    )
    .reset_index()
)

In [64]:
# Criação das razões agregadas
previous_by_client[
    "PREV_APPROVED_RATIO"
] = safe_divide(
    previous_by_client[
        "PREV_APPROVED_COUNT"
    ],
    previous_by_client[
        "PREV_APPLICATION_COUNT"
    ],
)

previous_by_client[
    "PREV_REFUSED_RATIO"
] = safe_divide(
    previous_by_client[
        "PREV_REFUSED_COUNT"
    ],
    previous_by_client[
        "PREV_APPLICATION_COUNT"
    ],
)

previous_by_client[
    "PREV_CANCELED_RATIO"
] = safe_divide(
    previous_by_client[
        "PREV_CANCELED_COUNT"
    ],
    previous_by_client[
        "PREV_APPLICATION_COUNT"
    ],
)

previous_by_client[
    "PREV_CASH_LOAN_RATIO"
] = safe_divide(
    previous_by_client[
        "PREV_CASH_LOAN_COUNT"
    ],
    previous_by_client[
        "PREV_APPLICATION_COUNT"
    ],
)

previous_by_client[
    "PREV_CONSUMER_LOAN_RATIO"
] = safe_divide(
    previous_by_client[
        "PREV_CONSUMER_LOAN_COUNT"
    ],
    previous_by_client[
        "PREV_APPLICATION_COUNT"
    ],
)

previous_by_client[
    "PREV_MOST_RECENT_APPROVED_AGE_YEARS"
] = (
    -previous_by_client[
        "PREV_MOST_RECENT_APPROVED_DECISION"
    ]
    / 365.25
)

In [65]:
# Validação da agregação de previous_application
n_previous_clients = (
    previous_application[
        "SK_ID_CURR"
    ]
    .nunique()
)

n_previous_aggregated_clients = (
    previous_by_client[
        "SK_ID_CURR"
    ]
    .nunique()
)

if (
    n_previous_clients
    != n_previous_aggregated_clients
):
    raise ValueError(
        "Quantidade de clientes divergente "
        "após a agregação."
    )

if previous_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "previous_by_client possui "
        "clientes duplicados."
    )

print(
    "Agregação validada:",
    f"{n_previous_aggregated_clients:,}",
    "clientes.",
)

print(
    "Quantidade de features:",
    previous_by_client.shape[1] - 1,
)

Agregação validada: 338,857 clientes.
Quantidade de features: 47


In [66]:
# Persistência das features de aplicações anteriores
previous_by_client_output = (
    paths.data_interim
    / "previous_application_by_client.parquet"
)

previous_by_client.to_parquet(
    previous_by_client_output,
    index=False,
)

previous_status_distribution.to_csv(
    paths.metadata
    / "previous_application_status_distribution.csv",
    index=False,
)

previous_special_values.to_csv(
    paths.metadata
    / "previous_application_special_values.csv",
    index=False,
)

print(
    "Features salvas em:",
    previous_by_client_output,
)

print(
    "Tamanho:",
    f"{previous_by_client_output.stat().st_size / 1024**2:,.2f} MB",
)

Features salvas em: /content/drive/MyDrive/home-credit-default-risk/data/interim/previous_application_by_client.parquet
Tamanho: 35.88 MB


In [67]:
# Liberação da tabela de aplicações anteriores
release_memory(
    "previous_application",
    "previous_by_client",
)

print(
    "Dados de previous_application "
    "removidos da memória."
)

Dados de previous_application removidos da memória.


## Processamento de `installments_payments`

A tabela `installments_payments` contém os pagamentos associados às parcelas dos contratos anteriores.

Uma mesma parcela pode possuir mais de um registro de pagamento, por exemplo em situações de pagamento parcial. Por isso, o processamento será realizado em três níveis:

1. consolidação dos registros de pagamento por parcela;
2. agregação das parcelas por contrato anterior;
3. agregação dos contratos por cliente.

Para evitar data leakage, as métricas comportamentais considerarão somente parcelas com vencimento até a data da aplicação atual:

$$
\texttt{DAYS\_INSTALMENT} \leq 0
$$

Pagamentos com `DAYS_ENTRY_PAYMENT > 0` representam movimentos posteriores à aplicação atual e não serão utilizados.

In [68]:
# Definição do caminho de installments_payments
INSTALLMENTS_PATH = (
    paths.data_raw
    / "installments_payments.csv"
)

INSTALLMENTS_COLUMNS = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT",
]

INSTALLMENTS_DTYPES = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "NUM_INSTALMENT_VERSION": "float32",
    "NUM_INSTALMENT_NUMBER": "int16",
    "DAYS_INSTALMENT": "float32",
    "DAYS_ENTRY_PAYMENT": "float32",
    "AMT_INSTALMENT": "float32",
    "AMT_PAYMENT": "float32",
}

In [69]:
# Carregamento de installments_payments
installments = pd.read_csv(
    INSTALLMENTS_PATH,
    usecols=INSTALLMENTS_COLUMNS,
    dtype=INSTALLMENTS_DTYPES,
)

print(
    "Dimensões:",
    installments.shape,
)

print(
    "Memória utilizada:",
    f"{installments.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(installments.head())

Dimensões: (13605401, 8)
Memória utilizada: 389.25 MB


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0000,6,"-1,180.0000","-1,187.0000","6,948.3599","6,948.3599"
1,1330831,151639,0.0000,34,"-2,156.0000","-2,156.0000","1,716.5250","1,716.5250"
2,2085231,193053,2.0000,1,-63.0000,-63.0000,"25,425.0000","25,425.0000"
3,2452527,199697,1.0000,3,"-2,418.0000","-2,426.0000","24,350.1309","24,350.1309"
4,2714724,167756,1.0000,2,"-1,383.0000","-1,366.0000","2,165.0400","2,160.5850"


In [70]:
# Validação estrutural de installments_payments
installments_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(installments),
        },
        {
            "validation": "Clientes",
            "value": (
                installments[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
        },
        {
            "validation": "Contratos",
            "value": (
                installments[
                    "SK_ID_PREV"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_CURR ausente"
            ),
            "value": (
                installments[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_PREV ausente"
            ),
            "value": (
                installments[
                    "SK_ID_PREV"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "DAYS_INSTALMENT ausente"
            ),
            "value": (
                installments[
                    "DAYS_INSTALMENT"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "DAYS_ENTRY_PAYMENT positivo"
            ),
            "value": (
                installments[
                    "DAYS_ENTRY_PAYMENT"
                ]
                .gt(0)
                .sum()
            ),
        },
        {
            "validation": (
                "Parcela com vencimento futuro"
            ),
            "value": (
                installments[
                    "DAYS_INSTALMENT"
                ]
                .gt(0)
                .sum()
            ),
        },
    ]
)

display(installments_validation)

,validation,value
0,Registros,13605401
1,Clientes,339587
2,Contratos,997752
3,SK_ID_CURR ausente,0
4,SK_ID_PREV ausente,0
5,DAYS_INSTALMENT ausente,0
6,DAYS_ENTRY_PAYMENT positivo,0
7,Parcela com vencimento futuro,0


In [71]:
# Validação da relação entre contrato e cliente
clients_per_installment_contract = (
    installments
    .groupby("SK_ID_PREV")[
        "SK_ID_CURR"
    ]
    .nunique()
)

inconsistent_installment_contracts = (
    clients_per_installment_contract
    .gt(1)
    .sum()
)

if inconsistent_installment_contracts > 0:
    raise ValueError(
        "Existem contratos associados a "
        "mais de um SK_ID_CURR: "
        f"{inconsistent_installment_contracts}"
    )

print(
    "Relação SK_ID_PREV -> SK_ID_CURR "
    "validada."
)

Relação SK_ID_PREV -> SK_ID_CURR validada.


In [72]:
# Identificação dos registros temporalmente válidos
installment_due_mask = (
    installments[
        "DAYS_INSTALMENT"
    ]
    .le(0)
)

payment_available_mask = (
    installments[
        "DAYS_ENTRY_PAYMENT"
    ]
    .le(0)
    |
    installments[
        "DAYS_ENTRY_PAYMENT"
    ]
    .isna()
)

installment_temporal_filter = (
    installment_due_mask
    & payment_available_mask
)

In [73]:
# Resumo do filtro temporal
installment_temporal_summary = pd.DataFrame(
    [
        {
            "group": "Registros originais",
            "n_records": len(installments),
        },
        {
            "group": (
                "Parcelas com vencimento futuro"
            ),
            "n_records": (
                (~installment_due_mask).sum()
            ),
        },
        {
            "group": (
                "Pagamentos posteriores à aplicação"
            ),
            "n_records": (
                (
                    installments[
                        "DAYS_ENTRY_PAYMENT"
                    ]
                    .gt(0)
                )
                .sum()
            ),
        },
        {
            "group": (
                "Registros utilizados"
            ),
            "n_records": (
                installment_temporal_filter.sum()
            ),
        },
    ]
)

installment_temporal_summary[
    "proportion"
] = (
    installment_temporal_summary[
        "n_records"
    ]
    / len(installments)
)

display(
    installment_temporal_summary.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,group,n_records,proportion
0,Registros originais,"13,605,401",100.00%
1,Parcelas com vencimento futuro,0,0.00%
2,Pagamentos posteriores à aplicação,0,0.00%
3,Registros utilizados,"13,605,401",100.00%


In [74]:
# Aplicação do filtro temporal
installments_available = (
    installments.loc[
        installment_temporal_filter
    ]
    .copy()
)

print(
    "Registros disponíveis:",
    installments_available.shape,
)

Registros disponíveis: (13605401, 8)


In [75]:
# Consolidação dos pagamentos por parcela
installments_by_installment = (
    installments_available
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "NUM_INSTALMENT_VERSION",
            "NUM_INSTALMENT_NUMBER",
        ],
        observed=True,
    )
    .agg(
        INST_DAYS_DUE=(
            "DAYS_INSTALMENT",
            "min",
        ),
        INST_FIRST_PAYMENT_DAY=(
            "DAYS_ENTRY_PAYMENT",
            "min",
        ),
        INST_LAST_PAYMENT_DAY=(
            "DAYS_ENTRY_PAYMENT",
            "max",
        ),
        INST_SCHEDULED_AMOUNT=(
            "AMT_INSTALMENT",
            "max",
        ),
        INST_PAID_AMOUNT=(
            "AMT_PAYMENT",
            "sum",
        ),
        INST_PAYMENT_RECORD_COUNT=(
            "AMT_PAYMENT",
            "size",
        ),
        INST_OBSERVED_PAYMENT_COUNT=(
            "AMT_PAYMENT",
            "count",
        ),
    )
    .reset_index()
)

print(
    "Parcelas consolidadas:",
    installments_by_installment.shape,
)

Parcelas consolidadas: (12951918, 11)


In [76]:
# Cálculo dos dias de atraso
installments_by_installment[
    "INST_DPD"
] = (
    installments_by_installment[
        "INST_LAST_PAYMENT_DAY"
    ]
    - installments_by_installment[
        "INST_DAYS_DUE"
    ]
).clip(lower=0)

installments_by_installment[
    "INST_DBD"
] = (
    installments_by_installment[
        "INST_DAYS_DUE"
    ]
    - installments_by_installment[
        "INST_FIRST_PAYMENT_DAY"
    ]
).clip(lower=0)

In [77]:
# Cálculo da diferença e razão de pagamento
installments_by_installment[
    "INST_PAYMENT_DIFF"
] = (
    installments_by_installment[
        "INST_SCHEDULED_AMOUNT"
    ]
    - installments_by_installment[
        "INST_PAID_AMOUNT"
    ]
)

installments_by_installment[
    "INST_PAYMENT_RATIO"
] = safe_divide(
    installments_by_installment[
        "INST_PAID_AMOUNT"
    ],
    installments_by_installment[
        "INST_SCHEDULED_AMOUNT"
    ],
)

In [78]:
# Criação das flags por parcela
installments_by_installment[
    "INST_IS_LATE"
] = (
    installments_by_installment[
        "INST_DPD"
    ]
    .gt(0)
    .astype("int8")
)

installments_by_installment[
    "INST_IS_SEVERELY_LATE"
] = (
    installments_by_installment[
        "INST_DPD"
    ]
    .gt(30)
    .astype("int8")
)

installments_by_installment[
    "INST_IS_UNDERPAID"
] = (
    installments_by_installment[
        "INST_PAYMENT_DIFF"
    ]
    .gt(0.01)
    .astype("int8")
)

installments_by_installment[
    "INST_IS_OVERPAID"
] = (
    installments_by_installment[
        "INST_PAYMENT_DIFF"
    ]
    .lt(-0.01)
    .astype("int8")
)

installments_by_installment[
    "INST_PAYMENT_MISSING"
] = (
    installments_by_installment[
        "INST_OBSERVED_PAYMENT_COUNT"
    ]
    .eq(0)
    .astype("int8")
)

In [79]:
# Agregação das parcelas por contrato
installments_by_contract = (
    installments_by_installment
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
        ],
        observed=True,
    )
    .agg(
        INST_CONTRACT_INSTALLMENT_COUNT=(
            "NUM_INSTALMENT_NUMBER",
            "size",
        ),
        INST_CONTRACT_PAYMENT_RECORD_COUNT=(
            "INST_PAYMENT_RECORD_COUNT",
            "sum",
        ),
        INST_CONTRACT_LATE_COUNT=(
            "INST_IS_LATE",
            "sum",
        ),
        INST_CONTRACT_SEVERE_LATE_COUNT=(
            "INST_IS_SEVERELY_LATE",
            "sum",
        ),
        INST_CONTRACT_UNDERPAID_COUNT=(
            "INST_IS_UNDERPAID",
            "sum",
        ),
        INST_CONTRACT_OVERPAID_COUNT=(
            "INST_IS_OVERPAID",
            "sum",
        ),
        INST_CONTRACT_MISSING_PAYMENT_COUNT=(
            "INST_PAYMENT_MISSING",
            "sum",
        ),
        INST_CONTRACT_DPD_MEAN=(
            "INST_DPD",
            "mean",
        ),
        INST_CONTRACT_DPD_MAX=(
            "INST_DPD",
            "max",
        ),
        INST_CONTRACT_DBD_MEAN=(
            "INST_DBD",
            "mean",
        ),
        INST_CONTRACT_DBD_MAX=(
            "INST_DBD",
            "max",
        ),
        INST_CONTRACT_SCHEDULED_SUM=(
            "INST_SCHEDULED_AMOUNT",
            "sum",
        ),
        INST_CONTRACT_PAID_SUM=(
            "INST_PAID_AMOUNT",
            "sum",
        ),
        INST_CONTRACT_PAYMENT_DIFF_SUM=(
            "INST_PAYMENT_DIFF",
            "sum",
        ),
        INST_CONTRACT_PAYMENT_RATIO_MEAN=(
            "INST_PAYMENT_RATIO",
            "mean",
        ),
        INST_CONTRACT_FIRST_DUE_DAY=(
            "INST_DAYS_DUE",
            "min",
        ),
        INST_CONTRACT_LAST_DUE_DAY=(
            "INST_DAYS_DUE",
            "max",
        ),
        INST_CONTRACT_LAST_PAYMENT_DAY=(
            "INST_LAST_PAYMENT_DAY",
            "max",
        ),
    )
    .reset_index()
)

In [80]:
# Criação das razões e flags por contrato
installments_by_contract[
    "INST_CONTRACT_LATE_RATIO"
] = safe_divide(
    installments_by_contract[
        "INST_CONTRACT_LATE_COUNT"
    ],
    installments_by_contract[
        "INST_CONTRACT_INSTALLMENT_COUNT"
    ],
)

installments_by_contract[
    "INST_CONTRACT_UNDERPAID_RATIO"
] = safe_divide(
    installments_by_contract[
        "INST_CONTRACT_UNDERPAID_COUNT"
    ],
    installments_by_contract[
        "INST_CONTRACT_INSTALLMENT_COUNT"
    ],
)

installments_by_contract[
    "INST_CONTRACT_TOTAL_PAYMENT_RATIO"
] = safe_divide(
    installments_by_contract[
        "INST_CONTRACT_PAID_SUM"
    ],
    installments_by_contract[
        "INST_CONTRACT_SCHEDULED_SUM"
    ],
)

installments_by_contract[
    "INST_CONTRACT_HAS_LATE"
] = (
    installments_by_contract[
        "INST_CONTRACT_LATE_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

installments_by_contract[
    "INST_CONTRACT_HAS_SEVERE_LATE"
] = (
    installments_by_contract[
        "INST_CONTRACT_SEVERE_LATE_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

In [81]:
# Agregação dos contratos por cliente
installments_by_client = (
    installments_by_contract
    .groupby(
        "SK_ID_CURR",
        observed=True,
    )
    .agg(
        INST_CONTRACT_COUNT=(
            "SK_ID_PREV",
            "nunique",
        ),
        INST_INSTALLMENT_COUNT=(
            "INST_CONTRACT_INSTALLMENT_COUNT",
            "sum",
        ),
        INST_PAYMENT_RECORD_COUNT=(
            "INST_CONTRACT_PAYMENT_RECORD_COUNT",
            "sum",
        ),
        INST_LATE_COUNT=(
            "INST_CONTRACT_LATE_COUNT",
            "sum",
        ),
        INST_SEVERE_LATE_COUNT=(
            "INST_CONTRACT_SEVERE_LATE_COUNT",
            "sum",
        ),
        INST_UNDERPAID_COUNT=(
            "INST_CONTRACT_UNDERPAID_COUNT",
            "sum",
        ),
        INST_OVERPAID_COUNT=(
            "INST_CONTRACT_OVERPAID_COUNT",
            "sum",
        ),
        INST_MISSING_PAYMENT_COUNT=(
            "INST_CONTRACT_MISSING_PAYMENT_COUNT",
            "sum",
        ),
        INST_CONTRACT_WITH_LATE_COUNT=(
            "INST_CONTRACT_HAS_LATE",
            "sum",
        ),
        INST_CONTRACT_WITH_SEVERE_LATE_COUNT=(
            "INST_CONTRACT_HAS_SEVERE_LATE",
            "sum",
        ),
        INST_DPD_MEAN=(
            "INST_CONTRACT_DPD_MEAN",
            "mean",
        ),
        INST_DPD_MAX=(
            "INST_CONTRACT_DPD_MAX",
            "max",
        ),
        INST_DBD_MEAN=(
            "INST_CONTRACT_DBD_MEAN",
            "mean",
        ),
        INST_DBD_MAX=(
            "INST_CONTRACT_DBD_MAX",
            "max",
        ),
        INST_SCHEDULED_AMOUNT_SUM=(
            "INST_CONTRACT_SCHEDULED_SUM",
            "sum",
        ),
        INST_PAID_AMOUNT_SUM=(
            "INST_CONTRACT_PAID_SUM",
            "sum",
        ),
        INST_PAYMENT_DIFF_SUM=(
            "INST_CONTRACT_PAYMENT_DIFF_SUM",
            "sum",
        ),
        INST_PAYMENT_RATIO_MEAN=(
            "INST_CONTRACT_PAYMENT_RATIO_MEAN",
            "mean",
        ),
        INST_LAST_DUE_DAY=(
            "INST_CONTRACT_LAST_DUE_DAY",
            "max",
        ),
        INST_LAST_PAYMENT_DAY=(
            "INST_CONTRACT_LAST_PAYMENT_DAY",
            "max",
        ),
    )
    .reset_index()
)

In [82]:
# Criação das razões de pagamento por cliente
installments_by_client[
    "INST_LATE_RATIO"
] = safe_divide(
    installments_by_client[
        "INST_LATE_COUNT"
    ],
    installments_by_client[
        "INST_INSTALLMENT_COUNT"
    ],
)

installments_by_client[
    "INST_SEVERE_LATE_RATIO"
] = safe_divide(
    installments_by_client[
        "INST_SEVERE_LATE_COUNT"
    ],
    installments_by_client[
        "INST_INSTALLMENT_COUNT"
    ],
)

installments_by_client[
    "INST_UNDERPAID_RATIO"
] = safe_divide(
    installments_by_client[
        "INST_UNDERPAID_COUNT"
    ],
    installments_by_client[
        "INST_INSTALLMENT_COUNT"
    ],
)

installments_by_client[
    "INST_CONTRACT_WITH_LATE_RATIO"
] = safe_divide(
    installments_by_client[
        "INST_CONTRACT_WITH_LATE_COUNT"
    ],
    installments_by_client[
        "INST_CONTRACT_COUNT"
    ],
)

installments_by_client[
    "INST_TOTAL_PAYMENT_RATIO"
] = safe_divide(
    installments_by_client[
        "INST_PAID_AMOUNT_SUM"
    ],
    installments_by_client[
        "INST_SCHEDULED_AMOUNT_SUM"
    ],
)

In [83]:
# Validação da agregação de installments
if installments_by_contract[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
    ]
].duplicated().any():
    raise ValueError(
        "Existem contratos duplicados na "
        "agregação intermediária."
    )

if installments_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "Existem clientes duplicados na "
        "agregação final."
    )

print(
    "Contratos agregados:",
    f"{len(installments_by_contract):,}",
)

print(
    "Clientes agregados:",
    f"{len(installments_by_client):,}",
)

print(
    "Features por cliente:",
    installments_by_client.shape[1] - 1,
)

Contratos agregados: 997,752
Clientes agregados: 339,587
Features por cliente: 25


In [84]:
# Persistência das agregações de installments
installments_by_contract_output = (
    paths.data_interim
    / "installments_by_contract.parquet"
)

installments_by_client_output = (
    paths.data_interim
    / "installments_by_client.parquet"
)

installments_by_contract.to_parquet(
    installments_by_contract_output,
    index=False,
)

installments_by_client.to_parquet(
    installments_by_client_output,
    index=False,
)

installment_temporal_summary.to_csv(
    paths.metadata
    / "installments_temporal_filter.csv",
    index=False,
)

print(
    "Contratos salvos em:",
    installments_by_contract_output,
)

print(
    "Clientes salvos em:",
    installments_by_client_output,
)

Contratos salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/installments_by_contract.parquet
Clientes salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/installments_by_client.parquet


In [85]:
# Liberação das tabelas de installments
release_memory(
    "installments",
    "installments_available",
    "installments_by_installment",
    "installments_by_contract",
    "installments_by_client",
)

print(
    "Tabelas de installments removidas "
    "da memória."
)

Tabelas de installments removidas da memória.


## Processamento de `credit_card_balance`

A tabela `credit_card_balance` contém observações mensais dos contratos anteriores de cartão de crédito.

As features buscarão representar:

- quantidade de contratos e meses observados;
- saldo médio e máximo;
- limite de crédito;
- utilização do limite;
- valores e frequência de saques e compras;
- pagamentos realizados;
- pagamento em relação ao mínimo;
- meses com atraso;
- intensidade máxima do atraso;
- utilização recente, considerando os últimos seis meses disponíveis.

O processamento será realizado primeiro por contrato (`SK_ID_PREV`) e depois por cliente (`SK_ID_CURR`).

In [86]:
# Definição do caminho de credit_card_balance
CREDIT_CARD_PATH = (
    paths.data_raw
    / "credit_card_balance.csv"
)

CREDIT_CARD_COLUMNS = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "MONTHS_BALANCE",
    "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "AMT_INST_MIN_REGULARITY",
    "AMT_PAYMENT_CURRENT",
    "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT",
    "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_OTHER_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT",
    "NAME_CONTRACT_STATUS",
    "SK_DPD",
    "SK_DPD_DEF",
]

CREDIT_CARD_DTYPES = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "MONTHS_BALANCE": "int16",
    "AMT_BALANCE": "float32",
    "AMT_CREDIT_LIMIT_ACTUAL": "float32",
    "AMT_DRAWINGS_ATM_CURRENT": "float32",
    "AMT_DRAWINGS_CURRENT": "float32",
    "AMT_DRAWINGS_OTHER_CURRENT": "float32",
    "AMT_DRAWINGS_POS_CURRENT": "float32",
    "AMT_INST_MIN_REGULARITY": "float32",
    "AMT_PAYMENT_CURRENT": "float32",
    "AMT_PAYMENT_TOTAL_CURRENT": "float32",
    "AMT_TOTAL_RECEIVABLE": "float32",
    "CNT_DRAWINGS_ATM_CURRENT": "float32",
    "CNT_DRAWINGS_CURRENT": "float32",
    "CNT_DRAWINGS_OTHER_CURRENT": "float32",
    "CNT_DRAWINGS_POS_CURRENT": "float32",
    "NAME_CONTRACT_STATUS": "category",
    "SK_DPD": "int32",
    "SK_DPD_DEF": "int32",
}

In [87]:
# Carregamento de credit_card_balance
credit_card = pd.read_csv(
    CREDIT_CARD_PATH,
    usecols=CREDIT_CARD_COLUMNS,
    dtype=CREDIT_CARD_DTYPES,
)

print(
    "Dimensões:",
    credit_card.shape,
)

print(
    "Memória utilizada:",
    f"{credit_card.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(credit_card.head())

Dimensões: (3840312, 20)
Memória utilizada: 274.68 MB


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.9700,"135,000.0000",0.0000,877.5000,0.0000,877.5000,"1,700.3250","1,800.0000","1,800.0000",0.0000,0.0000,1.0000,0.0000,1.0000,Active,0,0
1,2582071,363914,-1,"63,975.5547","45,000.0000","2,250.0000","2,250.0000",0.0000,0.0000,"2,250.0000","2,250.0000","2,250.0000","64,875.5547",1.0000,1.0000,0.0000,0.0000,Active,0,0
2,1740877,371185,-7,"31,815.2246","450,000.0000",0.0000,0.0000,0.0000,0.0000,"2,250.0000","2,250.0000","2,250.0000","31,460.0859",0.0000,0.0000,0.0000,0.0000,Active,0,0
3,1389973,337855,-4,"236,572.1094","225,000.0000","2,250.0000","2,250.0000",0.0000,0.0000,"11,795.7598","11,925.0000","11,925.0000","233,048.9688",1.0000,1.0000,0.0000,0.0000,Active,0,0
4,1891521,126868,-1,"453,919.4688","450,000.0000",0.0000,"11,547.0000",0.0000,"11,547.0000","22,924.8906","27,000.0000","27,000.0000","453,919.4688",0.0000,1.0000,0.0000,1.0000,Active,0,0


In [88]:
# Validação estrutural de credit_card_balance
credit_card_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(credit_card),
        },
        {
            "validation": "Clientes",
            "value": (
                credit_card[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
        },
        {
            "validation": "Contratos",
            "value": (
                credit_card[
                    "SK_ID_PREV"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_CURR ausente"
            ),
            "value": (
                credit_card[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_PREV ausente"
            ),
            "value": (
                credit_card[
                    "SK_ID_PREV"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "MONTHS_BALANCE positivo"
            ),
            "value": (
                credit_card[
                    "MONTHS_BALANCE"
                ]
                .gt(0)
                .sum()
            ),
        },
        {
            "validation": (
                "Pares contrato/mês duplicados"
            ),
            "value": (
                credit_card
                .duplicated(
                    subset=[
                        "SK_ID_PREV",
                        "MONTHS_BALANCE",
                    ]
                )
                .sum()
            ),
        },
    ]
)

display(credit_card_validation)

,validation,value
0,Registros,3840312
1,Clientes,103558
2,Contratos,104307
3,SK_ID_CURR ausente,0
4,SK_ID_PREV ausente,0
5,MONTHS_BALANCE positivo,0
6,Pares contrato/mês duplicados,0


In [89]:
# Interrupção se houver registros futuros
if credit_card[
    "MONTHS_BALANCE"
].gt(0).any():
    raise ValueError(
        "Existem saldos de cartão posteriores "
        "à aplicação atual."
    )

In [90]:
# Validação da relação entre contrato e cliente
credit_card_clients_per_contract = (
    credit_card
    .groupby("SK_ID_PREV")[
        "SK_ID_CURR"
    ]
    .nunique()
)

inconsistent_credit_card_contracts = (
    credit_card_clients_per_contract
    .gt(1)
    .sum()
)

if inconsistent_credit_card_contracts > 0:
    raise ValueError(
        "Existem contratos de cartão associados "
        "a mais de um cliente: "
        f"{inconsistent_credit_card_contracts}"
    )

print(
    "Relação SK_ID_PREV -> SK_ID_CURR "
    "validada."
)

Relação SK_ID_PREV -> SK_ID_CURR validada.


In [91]:
# Criação das razões mensais de cartão
credit_card[
    "CC_LIMIT_UTILIZATION"
] = safe_divide(
    credit_card[
        "AMT_BALANCE"
    ],
    credit_card[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ],
)

credit_card[
    "CC_DRAWINGS_LIMIT_RATIO"
] = safe_divide(
    credit_card[
        "AMT_DRAWINGS_CURRENT"
    ],
    credit_card[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ],
)

credit_card[
    "CC_PAYMENT_MIN_RATIO"
] = safe_divide(
    credit_card[
        "AMT_PAYMENT_TOTAL_CURRENT"
    ],
    credit_card[
        "AMT_INST_MIN_REGULARITY"
    ],
)

In [92]:
# Criação das flags mensais de cartão
credit_card[
    "CC_HAS_DPD"
] = (
    credit_card["SK_DPD"]
    .gt(0)
    .astype("int8")
)

credit_card[
    "CC_HAS_SEVERE_DPD"
] = (
    credit_card["SK_DPD"]
    .gt(30)
    .astype("int8")
)

credit_card[
    "CC_HAS_DPD_DEF"
] = (
    credit_card["SK_DPD_DEF"]
    .gt(0)
    .astype("int8")
)

credit_card[
    "CC_IS_OVER_LIMIT"
] = (
    credit_card[
        "AMT_BALANCE"
    ]
    .gt(
        credit_card[
            "AMT_CREDIT_LIMIT_ACTUAL"
        ]
    )
    .astype("int8")
)

credit_card[
    "CC_IS_ACTIVE"
] = (
    credit_card[
        "NAME_CONTRACT_STATUS"
    ]
    .astype("string")
    .eq("Active")
    .astype("int8")
)

In [95]:
# Definição da janela recente de seis meses
CREDIT_CARD_RECENT_MONTHS = 6

credit_card_recent_mask = (
    credit_card[
        "MONTHS_BALANCE"
    ]
    .ge(
        -CREDIT_CARD_RECENT_MONTHS
    )
)

credit_card[
    "CC_RECENT_BALANCE"
] = (
    credit_card[
        "AMT_BALANCE"
    ]
    .where(
        credit_card_recent_mask
    )
)

credit_card[
    "CC_RECENT_UTILIZATION"
] = (
    credit_card[
        "CC_LIMIT_UTILIZATION"
    ]
    .where(
        credit_card_recent_mask
    )
)

credit_card[
    "CC_RECENT_DPD"
] = (
    credit_card[
        "SK_DPD"
    ]
    .where(
        credit_card_recent_mask
    )
)

In [96]:
# Agregação mensal por contrato de cartão
credit_card_by_contract = (
    credit_card
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
        ],
        observed=True,
    )
    .agg(
        CC_CONTRACT_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "size",
        ),
        CC_CONTRACT_OLDEST_MONTH=(
            "MONTHS_BALANCE",
            "min",
        ),
        CC_CONTRACT_LATEST_MONTH=(
            "MONTHS_BALANCE",
            "max",
        ),
        CC_CONTRACT_BALANCE_MEAN=(
            "AMT_BALANCE",
            "mean",
        ),
        CC_CONTRACT_BALANCE_MAX=(
            "AMT_BALANCE",
            "max",
        ),
        CC_CONTRACT_LIMIT_MEAN=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "mean",
        ),
        CC_CONTRACT_LIMIT_MAX=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "max",
        ),
        CC_CONTRACT_UTILIZATION_MEAN=(
            "CC_LIMIT_UTILIZATION",
            "mean",
        ),
        CC_CONTRACT_UTILIZATION_MAX=(
            "CC_LIMIT_UTILIZATION",
            "max",
        ),
        CC_CONTRACT_DRAWINGS_SUM=(
            "AMT_DRAWINGS_CURRENT",
            "sum",
        ),
        CC_CONTRACT_DRAWINGS_MEAN=(
            "AMT_DRAWINGS_CURRENT",
            "mean",
        ),
        CC_CONTRACT_ATM_DRAWINGS_SUM=(
            "AMT_DRAWINGS_ATM_CURRENT",
            "sum",
        ),
        CC_CONTRACT_POS_DRAWINGS_SUM=(
            "AMT_DRAWINGS_POS_CURRENT",
            "sum",
        ),
        CC_CONTRACT_DRAWING_COUNT_SUM=(
            "CNT_DRAWINGS_CURRENT",
            "sum",
        ),
        CC_CONTRACT_PAYMENT_SUM=(
            "AMT_PAYMENT_TOTAL_CURRENT",
            "sum",
        ),
        CC_CONTRACT_PAYMENT_MEAN=(
            "AMT_PAYMENT_TOTAL_CURRENT",
            "mean",
        ),
        CC_CONTRACT_MIN_INSTALLMENT_SUM=(
            "AMT_INST_MIN_REGULARITY",
            "sum",
        ),
        CC_CONTRACT_PAYMENT_MIN_RATIO_MEAN=(
            "CC_PAYMENT_MIN_RATIO",
            "mean",
        ),
        CC_CONTRACT_RECEIVABLE_MEAN=(
            "AMT_TOTAL_RECEIVABLE",
            "mean",
        ),
        CC_CONTRACT_RECEIVABLE_MAX=(
            "AMT_TOTAL_RECEIVABLE",
            "max",
        ),
        CC_CONTRACT_DPD_MONTH_COUNT=(
            "CC_HAS_DPD",
            "sum",
        ),
        CC_CONTRACT_SEVERE_DPD_MONTH_COUNT=(
            "CC_HAS_SEVERE_DPD",
            "sum",
        ),
        CC_CONTRACT_DPD_DEF_MONTH_COUNT=(
            "CC_HAS_DPD_DEF",
            "sum",
        ),
        CC_CONTRACT_OVER_LIMIT_MONTH_COUNT=(
            "CC_IS_OVER_LIMIT",
            "sum",
        ),
        CC_CONTRACT_ACTIVE_MONTH_COUNT=(
            "CC_IS_ACTIVE",
            "sum",
        ),
        CC_CONTRACT_DPD_MEAN=(
            "SK_DPD",
            "mean",
        ),
        CC_CONTRACT_DPD_MAX=(
            "SK_DPD",
            "max",
        ),
        CC_CONTRACT_DPD_DEF_MAX=(
            "SK_DPD_DEF",
            "max",
        ),
        CC_CONTRACT_RECENT_BALANCE_MEAN=(
            "CC_RECENT_BALANCE",
            "mean",
        ),
        CC_CONTRACT_RECENT_UTILIZATION_MEAN=(
            "CC_RECENT_UTILIZATION",
            "mean",
        ),
        CC_CONTRACT_RECENT_UTILIZATION_MAX=(
            "CC_RECENT_UTILIZATION",
            "max",
        ),
        CC_CONTRACT_RECENT_DPD_MAX=(
            "CC_RECENT_DPD",
            "max",
        ),
    )
    .reset_index()
)

In [97]:
# Criação das razões por contrato de cartão
credit_card_by_contract[
    "CC_CONTRACT_DPD_MONTH_RATIO"
] = safe_divide(
    credit_card_by_contract[
        "CC_CONTRACT_DPD_MONTH_COUNT"
    ],
    credit_card_by_contract[
        "CC_CONTRACT_MONTH_COUNT"
    ],
)

credit_card_by_contract[
    "CC_CONTRACT_SEVERE_DPD_RATIO"
] = safe_divide(
    credit_card_by_contract[
        "CC_CONTRACT_SEVERE_DPD_MONTH_COUNT"
    ],
    credit_card_by_contract[
        "CC_CONTRACT_MONTH_COUNT"
    ],
)

credit_card_by_contract[
    "CC_CONTRACT_OVER_LIMIT_RATIO"
] = safe_divide(
    credit_card_by_contract[
        "CC_CONTRACT_OVER_LIMIT_MONTH_COUNT"
    ],
    credit_card_by_contract[
        "CC_CONTRACT_MONTH_COUNT"
    ],
)

credit_card_by_contract[
    "CC_CONTRACT_PAYMENT_MIN_TOTAL_RATIO"
] = safe_divide(
    credit_card_by_contract[
        "CC_CONTRACT_PAYMENT_SUM"
    ],
    credit_card_by_contract[
        "CC_CONTRACT_MIN_INSTALLMENT_SUM"
    ],
)

credit_card_by_contract[
    "CC_CONTRACT_HAS_DPD"
] = (
    credit_card_by_contract[
        "CC_CONTRACT_DPD_MONTH_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

credit_card_by_contract[
    "CC_CONTRACT_HAS_SEVERE_DPD"
] = (
    credit_card_by_contract[
        "CC_CONTRACT_SEVERE_DPD_MONTH_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

In [98]:
# Agregação dos contratos de cartão por cliente
credit_card_by_client = (
    credit_card_by_contract
    .groupby(
        "SK_ID_CURR",
        observed=True,
    )
    .agg(
        CC_CONTRACT_COUNT=(
            "SK_ID_PREV",
            "nunique",
        ),
        CC_MONTH_COUNT=(
            "CC_CONTRACT_MONTH_COUNT",
            "sum",
        ),
        CC_BALANCE_MEAN=(
            "CC_CONTRACT_BALANCE_MEAN",
            "mean",
        ),
        CC_BALANCE_MAX=(
            "CC_CONTRACT_BALANCE_MAX",
            "max",
        ),
        CC_LIMIT_MEAN=(
            "CC_CONTRACT_LIMIT_MEAN",
            "mean",
        ),
        CC_LIMIT_MAX=(
            "CC_CONTRACT_LIMIT_MAX",
            "max",
        ),
        CC_UTILIZATION_MEAN=(
            "CC_CONTRACT_UTILIZATION_MEAN",
            "mean",
        ),
        CC_UTILIZATION_MAX=(
            "CC_CONTRACT_UTILIZATION_MAX",
            "max",
        ),
        CC_DRAWINGS_SUM=(
            "CC_CONTRACT_DRAWINGS_SUM",
            "sum",
        ),
        CC_ATM_DRAWINGS_SUM=(
            "CC_CONTRACT_ATM_DRAWINGS_SUM",
            "sum",
        ),
        CC_POS_DRAWINGS_SUM=(
            "CC_CONTRACT_POS_DRAWINGS_SUM",
            "sum",
        ),
        CC_DRAWING_COUNT_SUM=(
            "CC_CONTRACT_DRAWING_COUNT_SUM",
            "sum",
        ),
        CC_PAYMENT_SUM=(
            "CC_CONTRACT_PAYMENT_SUM",
            "sum",
        ),
        CC_MIN_INSTALLMENT_SUM=(
            "CC_CONTRACT_MIN_INSTALLMENT_SUM",
            "sum",
        ),
        CC_PAYMENT_MIN_RATIO_MEAN=(
            "CC_CONTRACT_PAYMENT_MIN_RATIO_MEAN",
            "mean",
        ),
        CC_RECEIVABLE_MEAN=(
            "CC_CONTRACT_RECEIVABLE_MEAN",
            "mean",
        ),
        CC_RECEIVABLE_MAX=(
            "CC_CONTRACT_RECEIVABLE_MAX",
            "max",
        ),
        CC_DPD_MONTH_COUNT=(
            "CC_CONTRACT_DPD_MONTH_COUNT",
            "sum",
        ),
        CC_SEVERE_DPD_MONTH_COUNT=(
            "CC_CONTRACT_SEVERE_DPD_MONTH_COUNT",
            "sum",
        ),
        CC_DPD_DEF_MONTH_COUNT=(
            "CC_CONTRACT_DPD_DEF_MONTH_COUNT",
            "sum",
        ),
        CC_OVER_LIMIT_MONTH_COUNT=(
            "CC_CONTRACT_OVER_LIMIT_MONTH_COUNT",
            "sum",
        ),
        CC_CONTRACT_WITH_DPD_COUNT=(
            "CC_CONTRACT_HAS_DPD",
            "sum",
        ),
        CC_CONTRACT_WITH_SEVERE_DPD_COUNT=(
            "CC_CONTRACT_HAS_SEVERE_DPD",
            "sum",
        ),
        CC_DPD_MEAN=(
            "CC_CONTRACT_DPD_MEAN",
            "mean",
        ),
        CC_DPD_MAX=(
            "CC_CONTRACT_DPD_MAX",
            "max",
        ),
        CC_DPD_DEF_MAX=(
            "CC_CONTRACT_DPD_DEF_MAX",
            "max",
        ),
        CC_RECENT_BALANCE_MEAN=(
            "CC_CONTRACT_RECENT_BALANCE_MEAN",
            "mean",
        ),
        CC_RECENT_UTILIZATION_MEAN=(
            "CC_CONTRACT_RECENT_UTILIZATION_MEAN",
            "mean",
        ),
        CC_RECENT_UTILIZATION_MAX=(
            "CC_CONTRACT_RECENT_UTILIZATION_MAX",
            "max",
        ),
        CC_RECENT_DPD_MAX=(
            "CC_CONTRACT_RECENT_DPD_MAX",
            "max",
        ),
    )
    .reset_index()
)

In [99]:
# Criação das razões de cartão por cliente
credit_card_by_client[
    "CC_DPD_MONTH_RATIO"
] = safe_divide(
    credit_card_by_client[
        "CC_DPD_MONTH_COUNT"
    ],
    credit_card_by_client[
        "CC_MONTH_COUNT"
    ],
)

credit_card_by_client[
    "CC_SEVERE_DPD_MONTH_RATIO"
] = safe_divide(
    credit_card_by_client[
        "CC_SEVERE_DPD_MONTH_COUNT"
    ],
    credit_card_by_client[
        "CC_MONTH_COUNT"
    ],
)

credit_card_by_client[
    "CC_OVER_LIMIT_MONTH_RATIO"
] = safe_divide(
    credit_card_by_client[
        "CC_OVER_LIMIT_MONTH_COUNT"
    ],
    credit_card_by_client[
        "CC_MONTH_COUNT"
    ],
)

credit_card_by_client[
    "CC_TOTAL_PAYMENT_MIN_RATIO"
] = safe_divide(
    credit_card_by_client[
        "CC_PAYMENT_SUM"
    ],
    credit_card_by_client[
        "CC_MIN_INSTALLMENT_SUM"
    ],
)

credit_card_by_client[
    "CC_CONTRACT_WITH_DPD_RATIO"
] = safe_divide(
    credit_card_by_client[
        "CC_CONTRACT_WITH_DPD_COUNT"
    ],
    credit_card_by_client[
        "CC_CONTRACT_COUNT"
    ],
)

In [100]:
# Validação das agregações de cartão
if credit_card_by_contract[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
    ]
].duplicated().any():
    raise ValueError(
        "Existem contratos duplicados."
    )

if credit_card_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "Existem clientes duplicados."
    )

print(
    "Contratos agregados:",
    f"{len(credit_card_by_contract):,}",
)

print(
    "Clientes agregados:",
    f"{len(credit_card_by_client):,}",
)

print(
    "Features por cliente:",
    credit_card_by_client.shape[1] - 1,
)

Contratos agregados: 104,307
Clientes agregados: 103,558
Features por cliente: 35


In [101]:
# Persistência das agregações de cartão
credit_card_by_contract_output = (
    paths.data_interim
    / "credit_card_by_contract.parquet"
)

credit_card_by_client_output = (
    paths.data_interim
    / "credit_card_by_client.parquet"
)

credit_card_by_contract.to_parquet(
    credit_card_by_contract_output,
    index=False,
)

credit_card_by_client.to_parquet(
    credit_card_by_client_output,
    index=False,
)

print(
    "Contratos salvos em:",
    credit_card_by_contract_output,
)

print(
    "Clientes salvos em:",
    credit_card_by_client_output,
)

Contratos salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/credit_card_by_contract.parquet
Clientes salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/credit_card_by_client.parquet


In [102]:
# Liberação das tabelas de cartão
release_memory(
    "credit_card",
    "credit_card_by_contract",
    "credit_card_by_client",
)

print(
    "Tabelas de cartão removidas "
    "da memória."
)

Tabelas de cartão removidas da memória.


## Processamento de `POS_CASH_balance`

A tabela `POS_CASH_balance` contém observações mensais de contratos anteriores de crédito POS/CASH.

As features buscarão representar:

- quantidade de contratos e meses observados;
- quantidade de parcelas previstas e futuras;
- evolução aproximada do pagamento do contrato;
- meses com atraso;
- intensidade máxima do atraso;
- contratos com atrasos severos;
- contratos ativos e concluídos;
- comportamento recente nos últimos seis meses.

Após esta etapa, todas as tabelas auxiliares terão sido processadas.

In [103]:
# Definição do caminho de POS_CASH_balance
POS_CASH_PATH = (
    paths.data_raw
    / "POS_CASH_balance.csv"
)

POS_CASH_COLUMNS = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "MONTHS_BALANCE",
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE",
    "NAME_CONTRACT_STATUS",
    "SK_DPD",
    "SK_DPD_DEF",
]

POS_CASH_DTYPES = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "MONTHS_BALANCE": "int16",
    "CNT_INSTALMENT": "float32",
    "CNT_INSTALMENT_FUTURE": "float32",
    "NAME_CONTRACT_STATUS": "category",
    "SK_DPD": "int32",
    "SK_DPD_DEF": "int32",
}

In [104]:
# Carregamento de POS_CASH_balance
pos_cash = pd.read_csv(
    POS_CASH_PATH,
    usecols=POS_CASH_COLUMNS,
    dtype=POS_CASH_DTYPES,
)

print(
    "Dimensões:",
    pos_cash.shape,
)

print(
    "Memória utilizada:",
    f"{pos_cash.memory_usage(deep=True).sum() / 1024**2:,.2f} MB",
)

display(pos_cash.head())

Dimensões: (10001358, 8)
Memória utilizada: 257.53 MB


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0000,45.0000,Active,0,0
1,1715348,367990,-33,36.0000,35.0000,Active,0,0
2,1784872,397406,-32,12.0000,9.0000,Active,0,0
3,1903291,269225,-35,48.0000,42.0000,Active,0,0
4,2341044,334279,-35,36.0000,35.0000,Active,0,0


In [105]:
# Validação estrutural de POS_CASH_balance
pos_cash_validation = pd.DataFrame(
    [
        {
            "validation": "Registros",
            "value": len(pos_cash),
        },
        {
            "validation": "Clientes",
            "value": (
                pos_cash[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
        },
        {
            "validation": "Contratos",
            "value": (
                pos_cash[
                    "SK_ID_PREV"
                ]
                .nunique()
            ),
        },
        {
            "validation": (
                "SK_ID_CURR ausente"
            ),
            "value": (
                pos_cash[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "SK_ID_PREV ausente"
            ),
            "value": (
                pos_cash[
                    "SK_ID_PREV"
                ]
                .isna()
                .sum()
            ),
        },
        {
            "validation": (
                "MONTHS_BALANCE positivo"
            ),
            "value": (
                pos_cash[
                    "MONTHS_BALANCE"
                ]
                .gt(0)
                .sum()
            ),
        },
        {
            "validation": (
                "Pares contrato/mês duplicados"
            ),
            "value": (
                pos_cash
                .duplicated(
                    subset=[
                        "SK_ID_PREV",
                        "MONTHS_BALANCE",
                    ]
                )
                .sum()
            ),
        },
    ]
)

display(pos_cash_validation)

,validation,value
0,Registros,10001358
1,Clientes,337252
2,Contratos,936325
3,SK_ID_CURR ausente,0
4,SK_ID_PREV ausente,0
5,MONTHS_BALANCE positivo,0
6,Pares contrato/mês duplicados,0


In [106]:
# Interrupção se houver registros futuros
if pos_cash[
    "MONTHS_BALANCE"
].gt(0).any():
    raise ValueError(
        "Existem registros POS/CASH posteriores "
        "à aplicação atual."
    )

In [107]:
# Validação da relação entre contrato e cliente
pos_clients_per_contract = (
    pos_cash
    .groupby("SK_ID_PREV")[
        "SK_ID_CURR"
    ]
    .nunique()
)

inconsistent_pos_contracts = (
    pos_clients_per_contract
    .gt(1)
    .sum()
)

if inconsistent_pos_contracts > 0:
    raise ValueError(
        "Existem contratos POS/CASH associados "
        "a mais de um cliente: "
        f"{inconsistent_pos_contracts}"
    )

print(
    "Relação SK_ID_PREV -> SK_ID_CURR "
    "validada."
)

Relação SK_ID_PREV -> SK_ID_CURR validada.


In [108]:
# Distribuição de NAME_CONTRACT_STATUS
pos_status_distribution = (
    pos_cash[
        "NAME_CONTRACT_STATUS"
    ]
    .value_counts(
        dropna=False,
    )
    .rename_axis("contract_status")
    .reset_index(name="n_records")
)

pos_status_distribution[
    "proportion"
] = (
    pos_status_distribution[
        "n_records"
    ]
    / len(pos_cash)
)

display(
    pos_status_distribution.style.format(
        {
            "n_records": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,contract_status,n_records,proportion
0,Active,"9,151,119",91.50%
1,Completed,"744,883",7.45%
2,Signed,"87,260",0.87%
3,Demand,"7,065",0.07%
4,Returned to the store,"5,461",0.05%
5,Approved,"4,917",0.05%
6,Amortized debt,636,0.01%
7,Canceled,15,0.00%
8,XNA,2,0.00%


In [109]:
# Criação das razões mensais de POS/CASH
pos_cash[
    "POS_FUTURE_INSTALMENT_RATIO"
] = safe_divide(
    pos_cash[
        "CNT_INSTALMENT_FUTURE"
    ],
    pos_cash[
        "CNT_INSTALMENT"
    ],
)

pos_cash[
    "POS_COMPLETION_RATIO"
] = (
    1
    - pos_cash[
        "POS_FUTURE_INSTALMENT_RATIO"
    ]
)

In [110]:
# Criação das flags mensais de POS/CASH
pos_cash[
    "POS_HAS_DPD"
] = (
    pos_cash["SK_DPD"]
    .gt(0)
    .astype("int8")
)

pos_cash[
    "POS_HAS_SEVERE_DPD"
] = (
    pos_cash["SK_DPD"]
    .gt(30)
    .astype("int8")
)

pos_cash[
    "POS_HAS_DPD_DEF"
] = (
    pos_cash["SK_DPD_DEF"]
    .gt(0)
    .astype("int8")
)

pos_contract_status = (
    pos_cash[
        "NAME_CONTRACT_STATUS"
    ]
    .astype("string")
)

pos_cash[
    "POS_IS_ACTIVE"
] = (
    pos_contract_status
    .eq("Active")
    .astype("int8")
)

pos_cash[
    "POS_IS_COMPLETED"
] = (
    pos_contract_status
    .eq("Completed")
    .astype("int8")
)

In [111]:
# Definição da janela recente de POS/CASH
POS_RECENT_MONTHS = 6

pos_recent_mask = (
    pos_cash[
        "MONTHS_BALANCE"
    ]
    .ge(-POS_RECENT_MONTHS)
)

pos_cash[
    "POS_RECENT_DPD"
] = (
    pos_cash["SK_DPD"]
    .where(pos_recent_mask)
)

pos_cash[
    "POS_RECENT_DPD_DEF"
] = (
    pos_cash["SK_DPD_DEF"]
    .where(pos_recent_mask)
)

pos_cash[
    "POS_RECENT_FUTURE_INSTALMENTS"
] = (
    pos_cash[
        "CNT_INSTALMENT_FUTURE"
    ]
    .where(pos_recent_mask)
)

pos_cash[
    "POS_RECENT_COMPLETION_RATIO"
] = (
    pos_cash[
        "POS_COMPLETION_RATIO"
    ]
    .where(pos_recent_mask)
)

In [112]:
# Agregação mensal por contrato POS/CASH
pos_cash_by_contract = (
    pos_cash
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
        ],
        observed=True,
    )
    .agg(
        POS_CONTRACT_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "size",
        ),
        POS_CONTRACT_OLDEST_MONTH=(
            "MONTHS_BALANCE",
            "min",
        ),
        POS_CONTRACT_LATEST_MONTH=(
            "MONTHS_BALANCE",
            "max",
        ),
        POS_CONTRACT_INSTALMENT_MEAN=(
            "CNT_INSTALMENT",
            "mean",
        ),
        POS_CONTRACT_INSTALMENT_MAX=(
            "CNT_INSTALMENT",
            "max",
        ),
        POS_CONTRACT_FUTURE_MEAN=(
            "CNT_INSTALMENT_FUTURE",
            "mean",
        ),
        POS_CONTRACT_FUTURE_MIN=(
            "CNT_INSTALMENT_FUTURE",
            "min",
        ),
        POS_CONTRACT_FUTURE_MAX=(
            "CNT_INSTALMENT_FUTURE",
            "max",
        ),
        POS_CONTRACT_COMPLETION_MEAN=(
            "POS_COMPLETION_RATIO",
            "mean",
        ),
        POS_CONTRACT_COMPLETION_MAX=(
            "POS_COMPLETION_RATIO",
            "max",
        ),
        POS_CONTRACT_DPD_MONTH_COUNT=(
            "POS_HAS_DPD",
            "sum",
        ),
        POS_CONTRACT_SEVERE_DPD_MONTH_COUNT=(
            "POS_HAS_SEVERE_DPD",
            "sum",
        ),
        POS_CONTRACT_DPD_DEF_MONTH_COUNT=(
            "POS_HAS_DPD_DEF",
            "sum",
        ),
        POS_CONTRACT_ACTIVE_MONTH_COUNT=(
            "POS_IS_ACTIVE",
            "sum",
        ),
        POS_CONTRACT_COMPLETED_MONTH_COUNT=(
            "POS_IS_COMPLETED",
            "sum",
        ),
        POS_CONTRACT_DPD_MEAN=(
            "SK_DPD",
            "mean",
        ),
        POS_CONTRACT_DPD_MAX=(
            "SK_DPD",
            "max",
        ),
        POS_CONTRACT_DPD_DEF_MEAN=(
            "SK_DPD_DEF",
            "mean",
        ),
        POS_CONTRACT_DPD_DEF_MAX=(
            "SK_DPD_DEF",
            "max",
        ),
        POS_CONTRACT_RECENT_DPD_MEAN=(
            "POS_RECENT_DPD",
            "mean",
        ),
        POS_CONTRACT_RECENT_DPD_MAX=(
            "POS_RECENT_DPD",
            "max",
        ),
        POS_CONTRACT_RECENT_DPD_DEF_MAX=(
            "POS_RECENT_DPD_DEF",
            "max",
        ),
        POS_CONTRACT_RECENT_FUTURE_MEAN=(
            "POS_RECENT_FUTURE_INSTALMENTS",
            "mean",
        ),
        POS_CONTRACT_RECENT_COMPLETION_MEAN=(
            "POS_RECENT_COMPLETION_RATIO",
            "mean",
        ),
    )
    .reset_index()
)

In [113]:
# Criação das razões por contrato POS/CASH
pos_cash_by_contract[
    "POS_CONTRACT_DPD_MONTH_RATIO"
] = safe_divide(
    pos_cash_by_contract[
        "POS_CONTRACT_DPD_MONTH_COUNT"
    ],
    pos_cash_by_contract[
        "POS_CONTRACT_MONTH_COUNT"
    ],
)

pos_cash_by_contract[
    "POS_CONTRACT_SEVERE_DPD_RATIO"
] = safe_divide(
    pos_cash_by_contract[
        "POS_CONTRACT_SEVERE_DPD_MONTH_COUNT"
    ],
    pos_cash_by_contract[
        "POS_CONTRACT_MONTH_COUNT"
    ],
)

pos_cash_by_contract[
    "POS_CONTRACT_DPD_DEF_RATIO"
] = safe_divide(
    pos_cash_by_contract[
        "POS_CONTRACT_DPD_DEF_MONTH_COUNT"
    ],
    pos_cash_by_contract[
        "POS_CONTRACT_MONTH_COUNT"
    ],
)

pos_cash_by_contract[
    "POS_CONTRACT_HAS_DPD"
] = (
    pos_cash_by_contract[
        "POS_CONTRACT_DPD_MONTH_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

pos_cash_by_contract[
    "POS_CONTRACT_HAS_SEVERE_DPD"
] = (
    pos_cash_by_contract[
        "POS_CONTRACT_SEVERE_DPD_MONTH_COUNT"
    ]
    .gt(0)
    .astype("int8")
)

In [114]:
# Agregação dos contratos POS/CASH por cliente
pos_cash_by_client = (
    pos_cash_by_contract
    .groupby(
        "SK_ID_CURR",
        observed=True,
    )
    .agg(
        POS_CONTRACT_COUNT=(
            "SK_ID_PREV",
            "nunique",
        ),
        POS_MONTH_COUNT=(
            "POS_CONTRACT_MONTH_COUNT",
            "sum",
        ),
        POS_INSTALMENT_MEAN=(
            "POS_CONTRACT_INSTALMENT_MEAN",
            "mean",
        ),
        POS_INSTALMENT_MAX=(
            "POS_CONTRACT_INSTALMENT_MAX",
            "max",
        ),
        POS_FUTURE_INSTALMENT_MEAN=(
            "POS_CONTRACT_FUTURE_MEAN",
            "mean",
        ),
        POS_FUTURE_INSTALMENT_MIN=(
            "POS_CONTRACT_FUTURE_MIN",
            "min",
        ),
        POS_COMPLETION_RATIO_MEAN=(
            "POS_CONTRACT_COMPLETION_MEAN",
            "mean",
        ),
        POS_COMPLETION_RATIO_MAX=(
            "POS_CONTRACT_COMPLETION_MAX",
            "max",
        ),
        POS_DPD_MONTH_COUNT=(
            "POS_CONTRACT_DPD_MONTH_COUNT",
            "sum",
        ),
        POS_SEVERE_DPD_MONTH_COUNT=(
            "POS_CONTRACT_SEVERE_DPD_MONTH_COUNT",
            "sum",
        ),
        POS_DPD_DEF_MONTH_COUNT=(
            "POS_CONTRACT_DPD_DEF_MONTH_COUNT",
            "sum",
        ),
        POS_CONTRACT_WITH_DPD_COUNT=(
            "POS_CONTRACT_HAS_DPD",
            "sum",
        ),
        POS_CONTRACT_WITH_SEVERE_DPD_COUNT=(
            "POS_CONTRACT_HAS_SEVERE_DPD",
            "sum",
        ),
        POS_DPD_MEAN=(
            "POS_CONTRACT_DPD_MEAN",
            "mean",
        ),
        POS_DPD_MAX=(
            "POS_CONTRACT_DPD_MAX",
            "max",
        ),
        POS_DPD_DEF_MEAN=(
            "POS_CONTRACT_DPD_DEF_MEAN",
            "mean",
        ),
        POS_DPD_DEF_MAX=(
            "POS_CONTRACT_DPD_DEF_MAX",
            "max",
        ),
        POS_RECENT_DPD_MEAN=(
            "POS_CONTRACT_RECENT_DPD_MEAN",
            "mean",
        ),
        POS_RECENT_DPD_MAX=(
            "POS_CONTRACT_RECENT_DPD_MAX",
            "max",
        ),
        POS_RECENT_DPD_DEF_MAX=(
            "POS_CONTRACT_RECENT_DPD_DEF_MAX",
            "max",
        ),
        POS_RECENT_FUTURE_INSTALMENT_MEAN=(
            "POS_CONTRACT_RECENT_FUTURE_MEAN",
            "mean",
        ),
        POS_RECENT_COMPLETION_RATIO_MEAN=(
            "POS_CONTRACT_RECENT_COMPLETION_MEAN",
            "mean",
        ),
    )
    .reset_index()
)

In [115]:
# Criação das razões POS/CASH por cliente
pos_cash_by_client[
    "POS_DPD_MONTH_RATIO"
] = safe_divide(
    pos_cash_by_client[
        "POS_DPD_MONTH_COUNT"
    ],
    pos_cash_by_client[
        "POS_MONTH_COUNT"
    ],
)

pos_cash_by_client[
    "POS_SEVERE_DPD_MONTH_RATIO"
] = safe_divide(
    pos_cash_by_client[
        "POS_SEVERE_DPD_MONTH_COUNT"
    ],
    pos_cash_by_client[
        "POS_MONTH_COUNT"
    ],
)

pos_cash_by_client[
    "POS_DPD_DEF_MONTH_RATIO"
] = safe_divide(
    pos_cash_by_client[
        "POS_DPD_DEF_MONTH_COUNT"
    ],
    pos_cash_by_client[
        "POS_MONTH_COUNT"
    ],
)

pos_cash_by_client[
    "POS_CONTRACT_WITH_DPD_RATIO"
] = safe_divide(
    pos_cash_by_client[
        "POS_CONTRACT_WITH_DPD_COUNT"
    ],
    pos_cash_by_client[
        "POS_CONTRACT_COUNT"
    ],
)

In [116]:
# Validação das agregações POS/CASH
if pos_cash_by_contract[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
    ]
].duplicated().any():
    raise ValueError(
        "Existem contratos POS/CASH duplicados."
    )

if pos_cash_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "Existem clientes duplicados na "
        "agregação POS/CASH."
    )

print(
    "Contratos agregados:",
    f"{len(pos_cash_by_contract):,}",
)

print(
    "Clientes agregados:",
    f"{len(pos_cash_by_client):,}",
)

print(
    "Features por cliente:",
    pos_cash_by_client.shape[1] - 1,
)

Contratos agregados: 936,325
Clientes agregados: 337,252
Features por cliente: 26


In [117]:
# Persistência das agregações POS/CASH
pos_cash_by_contract_output = (
    paths.data_interim
    / "pos_cash_by_contract.parquet"
)

pos_cash_by_client_output = (
    paths.data_interim
    / "pos_cash_by_client.parquet"
)

pos_cash_by_contract.to_parquet(
    pos_cash_by_contract_output,
    index=False,
)

pos_cash_by_client.to_parquet(
    pos_cash_by_client_output,
    index=False,
)

pos_status_distribution.to_csv(
    paths.metadata
    / "pos_cash_status_distribution.csv",
    index=False,
)

print(
    "Contratos salvos em:",
    pos_cash_by_contract_output,
)

print(
    "Clientes salvos em:",
    pos_cash_by_client_output,
)

Contratos salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/pos_cash_by_contract.parquet
Clientes salvos em: /content/drive/MyDrive/home-credit-default-risk/data/interim/pos_cash_by_client.parquet


In [118]:
# Liberação das tabelas POS/CASH
release_memory(
    "pos_cash",
    "pos_cash_by_contract",
    "pos_cash_by_client",
)

print(
    "Tabelas POS/CASH removidas "
    "da memória."
)

Tabelas POS/CASH removidas da memória.


## Consolidação das Features Auxiliares

As agregações produzidas possuem uma linha por `SK_ID_CURR` e serão combinadas por meio de `outer join`.

O `outer join` preserva clientes que aparecem em qualquer uma das fontes históricas, mesmo que não possuam registros nas demais.

Para cada fonte será criada uma flag de disponibilidade:

- `HAS_BUREAU_HISTORY`;
- `HAS_PREVIOUS_APPLICATION_HISTORY`;
- `HAS_INSTALLMENTS_HISTORY`;
- `HAS_CREDIT_CARD_HISTORY`;
- `HAS_POS_CASH_HISTORY`.

A ausência de uma fonte será representada pela flag igual a zero. Features de contagem receberão zero quando não houver registros, enquanto médias, máximos, razões e valores permanecerão como missing.

In [119]:
# Definição dos arquivos agregados por cliente
auxiliary_feature_files = {
    "bureau": (
        paths.data_interim
        / "bureau_by_client.parquet"
    ),
    "previous_application": (
        paths.data_interim
        / "previous_application_by_client.parquet"
    ),
    "installments": (
        paths.data_interim
        / "installments_by_client.parquet"
    ),
    "credit_card": (
        paths.data_interim
        / "credit_card_by_client.parquet"
    ),
    "pos_cash": (
        paths.data_interim
        / "pos_cash_by_client.parquet"
    ),
}

missing_auxiliary_feature_files = [
    str(file_path)
    for file_path in (
        auxiliary_feature_files.values()
    )
    if not file_path.exists()
]

if missing_auxiliary_feature_files:
    raise FileNotFoundError(
        "Agregações auxiliares ausentes: "
        f"{missing_auxiliary_feature_files}"
    )

In [120]:
# Carregamento das agregações por cliente
auxiliary_feature_tables = {
    source_name: pd.read_parquet(
        file_path
    )
    for source_name, file_path
    in auxiliary_feature_files.items()
}

for source_name, dataframe in (
    auxiliary_feature_tables.items()
):
    print(
        source_name,
        dataframe.shape,
    )

bureau (305811, 53)
previous_application (338857, 48)
installments (339587, 26)
credit_card (103558, 36)
pos_cash (337252, 27)


In [121]:
# Validação das chaves das agregações
auxiliary_key_validation = []

for source_name, dataframe in (
    auxiliary_feature_tables.items()
):
    auxiliary_key_validation.append(
        {
            "source": source_name,
            "n_records": len(dataframe),
            "n_unique_clients": (
                dataframe[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
            "n_missing_ids": (
                dataframe[
                    "SK_ID_CURR"
                ]
                .isna()
                .sum()
            ),
            "n_duplicated_ids": (
                dataframe[
                    "SK_ID_CURR"
                ]
                .duplicated()
                .sum()
            ),
            "n_features": (
                dataframe.shape[1] - 1
            ),
        }
    )

auxiliary_key_validation = pd.DataFrame(
    auxiliary_key_validation
)

display(
    auxiliary_key_validation.style.format(
        {
            "n_records": "{:,.0f}",
            "n_unique_clients": "{:,.0f}",
            "n_missing_ids": "{:,.0f}",
            "n_duplicated_ids": "{:,.0f}",
            "n_features": "{:,.0f}",
        }
    )
)

,source,n_records,n_unique_clients,n_missing_ids,n_duplicated_ids,n_features
0,bureau,"305,811","305,811",0,0,52
1,previous_application,"338,857","338,857",0,0,47
2,installments,"339,587","339,587",0,0,25
3,credit_card,"103,558","103,558",0,0,35
4,pos_cash,"337,252","337,252",0,0,26


In [122]:
# Interrupção se alguma chave for inválida
if auxiliary_key_validation[
    "n_missing_ids"
].gt(0).any():
    raise ValueError(
        "Existem SK_ID_CURR ausentes nas "
        "agregações auxiliares."
    )

if auxiliary_key_validation[
    "n_duplicated_ids"
].gt(0).any():
    raise ValueError(
        "Existem SK_ID_CURR duplicados nas "
        "agregações auxiliares."
    )

In [123]:
# Validação de nomes duplicados entre fontes
feature_name_sources = {}

for source_name, dataframe in (
    auxiliary_feature_tables.items()
):
    for feature in dataframe.columns:
        if feature == "SK_ID_CURR":
            continue

        feature_name_sources.setdefault(
            feature,
            [],
        ).append(source_name)

duplicated_feature_names = {
    feature: sources
    for feature, sources
    in feature_name_sources.items()
    if len(sources) > 1
}

if duplicated_feature_names:
    raise ValueError(
        "Existem features com nomes repetidos "
        "entre as fontes: "
        f"{duplicated_feature_names}"
    )

print(
    "Não existem nomes de features "
    "duplicados entre as fontes."
)

Não existem nomes de features duplicados entre as fontes.


In [124]:
# Definição das flags de disponibilidade
source_availability_flags = {
    "bureau": "HAS_BUREAU_HISTORY",
    "previous_application": (
        "HAS_PREVIOUS_APPLICATION_HISTORY"
    ),
    "installments": (
        "HAS_INSTALLMENTS_HISTORY"
    ),
    "credit_card": (
        "HAS_CREDIT_CARD_HISTORY"
    ),
    "pos_cash": (
        "HAS_POS_CASH_HISTORY"
    ),
}

In [125]:
# Inclusão das flags nas tabelas
for source_name, dataframe in (
    auxiliary_feature_tables.items()
):
    availability_flag = (
        source_availability_flags[
            source_name
        ]
    )

    dataframe[
        availability_flag
    ] = np.int8(1)

In [126]:
# Consolidação das fontes históricas
auxiliary_features_by_client = None

for source_name, dataframe in (
    auxiliary_feature_tables.items()
):
    if auxiliary_features_by_client is None:
        auxiliary_features_by_client = (
            dataframe.copy()
        )

        continue

    auxiliary_features_by_client = (
        auxiliary_features_by_client
        .merge(
            dataframe,
            on="SK_ID_CURR",
            how="outer",
            validate="one_to_one",
        )
    )

print(
    "Dimensões consolidadas:",
    auxiliary_features_by_client.shape,
)

Dimensões consolidadas: (353895, 191)


In [127]:
# Preenchimento das flags de disponibilidade
availability_flag_columns = list(
    source_availability_flags.values()
)

auxiliary_features_by_client[
    availability_flag_columns
] = (
    auxiliary_features_by_client[
        availability_flag_columns
    ]
    .fillna(0)
    .astype("int8")
)

In [128]:
# Identificação das features de contagem
auxiliary_count_columns = [
    column
    for column in (
        auxiliary_features_by_client.columns
    )
    if column.endswith("_COUNT")
]

print(
    "Features de contagem:",
    len(auxiliary_count_columns),
)

Features de contagem: 50


In [129]:
# Preenchimento das contagens ausentes
auxiliary_features_by_client[
    auxiliary_count_columns
] = (
    auxiliary_features_by_client[
        auxiliary_count_columns
    ]
    .fillna(0)
)

In [130]:
# Criação da quantidade de fontes disponíveis
auxiliary_features_by_client[
    "AUX_AVAILABLE_SOURCE_COUNT"
] = (
    auxiliary_features_by_client[
        availability_flag_columns
    ]
    .sum(axis=1)
    .astype("int8")
)

In [131]:
# Validação da tabela histórica consolidada
if auxiliary_features_by_client[
    "SK_ID_CURR"
].isna().any():
    raise ValueError(
        "A tabela consolidada possui "
        "SK_ID_CURR ausente."
    )

if auxiliary_features_by_client[
    "SK_ID_CURR"
].duplicated().any():
    raise ValueError(
        "A tabela consolidada possui "
        "clientes duplicados."
    )

if "TARGET" in (
    auxiliary_features_by_client.columns
):
    raise ValueError(
        "TARGET foi incorporado indevidamente "
        "às features auxiliares."
    )

print(
    "Clientes consolidados:",
    f"{len(auxiliary_features_by_client):,}",
)

print(
    "Features históricas:",
    auxiliary_features_by_client.shape[1] - 1,
)

Clientes consolidados: 353,895
Features históricas: 191


In [132]:
# Construção do resumo de cobertura das fontes
auxiliary_source_coverage = pd.DataFrame(
    [
        {
            "source": source_name,
            "availability_flag": (
                availability_flag
            ),
            "n_clients": (
                auxiliary_features_by_client[
                    availability_flag
                ]
                .sum()
            ),
            "proportion_of_historical_union": (
                auxiliary_features_by_client[
                    availability_flag
                ]
                .mean()
            ),
        }
        for source_name, availability_flag
        in source_availability_flags.items()
    ]
)

display(
    auxiliary_source_coverage.style.format(
        {
            "n_clients": "{:,.0f}",
            "proportion_of_historical_union": (
                "{:.2%}"
            ),
        }
    )
)

,source,availability_flag,n_clients,proportion_of_historical_union
0,bureau,HAS_BUREAU_HISTORY,"305,811",86.41%
1,previous_application,HAS_PREVIOUS_APPLICATION_HISTORY,"338,857",95.75%
2,installments,HAS_INSTALLMENTS_HISTORY,"339,587",95.96%
3,credit_card,HAS_CREDIT_CARD_HISTORY,"103,558",29.26%
4,pos_cash,HAS_POS_CASH_HISTORY,"337,252",95.30%


In [133]:
# Distribuição da quantidade de fontes disponíveis
available_source_distribution = (
    auxiliary_features_by_client[
        "AUX_AVAILABLE_SOURCE_COUNT"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "available_source_count"
    )
    .reset_index(name="n_clients")
)

available_source_distribution[
    "proportion"
] = (
    available_source_distribution[
        "n_clients"
    ]
    / len(auxiliary_features_by_client)
)

display(
    available_source_distribution.style.format(
        {
            "n_clients": "{:,.0f}",
            "proportion": "{:.2%}",
        }
    )
)

,available_source_count,n_clients,proportion
0,1,"13,052",3.69%
1,2,"1,135",0.32%
2,3,"36,346",10.27%
3,4,"216,105",61.06%
4,5,"87,257",24.66%


In [134]:
# Construção do schema das features históricas
def identify_auxiliary_source(
    feature,
):
    if feature.startswith("BU_"):
        return "bureau"

    if feature.startswith("PREV_"):
        return "previous_application"

    if feature.startswith("INST_"):
        return "installments"

    if feature.startswith("CC_"):
        return "credit_card"

    if feature.startswith("POS_"):
        return "pos_cash"

    if feature.startswith("HAS_"):
        return "availability"

    if feature.startswith("AUX_"):
        return "availability"

    return "unclassified"


auxiliary_feature_schema = pd.DataFrame(
    {
        "feature": [
            column
            for column in (
                auxiliary_features_by_client
                .columns
            )
            if column != "SK_ID_CURR"
        ]
    }
)

auxiliary_feature_schema["source"] = (
    auxiliary_feature_schema[
        "feature"
    ]
    .map(
        identify_auxiliary_source
    )
)

auxiliary_feature_schema["dtype"] = (
    auxiliary_feature_schema[
        "feature"
    ]
    .map(
        lambda feature: str(
            auxiliary_features_by_client[
                feature
            ].dtype
        )
    )
)

auxiliary_feature_schema[
    "missing_rate_historical_union"
] = (
    auxiliary_feature_schema[
        "feature"
    ]
    .map(
        auxiliary_features_by_client
        .drop(
            columns=["SK_ID_CURR"]
        )
        .isna()
        .mean()
    )
)

display(
    auxiliary_feature_schema.style.format(
        {
            "missing_rate_historical_union": (
                "{:.2%}"
            ),
        }
    )
)

,feature,source,dtype,missing_rate_historical_union
0,BU_CONTRACT_COUNT,bureau,float64,0.00%
1,BU_CREDIT_TYPE_COUNT,bureau,float64,0.00%
2,BU_CURRENCY_COUNT,bureau,float64,0.00%
3,BU_ACTIVE_COUNT,bureau,float64,0.00%
4,BU_CLOSED_COUNT,bureau,float64,0.00%
5,BU_SOLD_COUNT,bureau,float64,0.00%
6,BU_BAD_DEBT_COUNT,bureau,float64,0.00%
7,BU_CURRENT_DEBT_COUNT,bureau,float64,0.00%
8,BU_OVERDUE_CONTRACT_COUNT,bureau,float64,0.00%
9,BU_BALANCE_HISTORY_COUNT,bureau,float64,0.00%


In [135]:
# Interrupção se alguma origem não for identificada
unclassified_auxiliary_features = (
    auxiliary_feature_schema.loc[
        auxiliary_feature_schema[
            "source"
        ]
        .eq("unclassified"),
        "feature",
    ]
    .tolist()
)

if unclassified_auxiliary_features:
    raise ValueError(
        "Features auxiliares sem origem "
        "identificada: "
        f"{unclassified_auxiliary_features}"
    )

In [136]:
# Persistência das features históricas consolidadas
auxiliary_features_output = (
    paths.data_interim
    / "auxiliary_features_by_client.parquet"
)

auxiliary_features_by_client.to_parquet(
    auxiliary_features_output,
    index=False,
)

auxiliary_feature_schema.to_csv(
    paths.metadata
    / "auxiliary_feature_schema.csv",
    index=False,
)

auxiliary_source_coverage.to_csv(
    paths.metadata
    / "auxiliary_source_coverage.csv",
    index=False,
)

available_source_distribution.to_csv(
    paths.metadata
    / "auxiliary_available_source_distribution.csv",
    index=False,
)

print(
    "Features consolidadas salvas em:",
    auxiliary_features_output,
)

print(
    "Tamanho:",
    f"{auxiliary_features_output.stat().st_size / 1024**2:,.2f} MB",
)

Features consolidadas salvas em: /content/drive/MyDrive/home-credit-default-risk/data/interim/auxiliary_features_by_client.parquet
Tamanho: 95.19 MB


In [137]:
# Liberação das agregações individuais
for source_name in list(
    auxiliary_feature_tables.keys()
):
    del auxiliary_feature_tables[
        source_name
    ]

del auxiliary_feature_tables

gc.collect()

print(
    "Agregações individuais removidas "
    "da memória."
)

Agregações individuais removidas da memória.


## Construção das Bases Finais de Modelagem

As features históricas serão combinadas com as bases preparadas no Notebook 03 por meio de `left join`.

A base de aplicações será sempre o lado esquerdo do merge. Assim:

- nenhum cliente das amostras será removido;
- nenhum novo cliente será incluído;
- o `TARGET` permanecerá inalterado;
- clientes sem histórico receberão flags de disponibilidade iguais a zero;
- contagens ausentes receberão zero;
- médias, máximos, valores e razões permanecerão como missing quando não houver histórico.

A base final combinará informações da aplicação atual com o histórico financeiro do cliente.

In [138]:
# Definição das bases produzidas pelo Notebook 03
application_prepared_files = {
    "train": (
        paths.data_processed
        / "application_train_prepared.parquet"
    ),
    "validation": (
        paths.data_processed
        / "application_validation_prepared.parquet"
    ),
    "holdout_test": (
        paths.data_processed
        / "application_holdout_test_prepared.parquet"
    ),
}

missing_application_files = [
    str(file_path)
    for file_path in (
        application_prepared_files.values()
    )
    if not file_path.exists()
]

if missing_application_files:
    raise FileNotFoundError(
        "Bases do Notebook 03 ausentes: "
        f"{missing_application_files}"
    )

In [139]:
# Carregamento das bases preparadas
application_prepared_samples = {
    sample_name: pd.read_parquet(
        file_path
    )
    for sample_name, file_path
    in application_prepared_files.items()
}

for sample_name, dataframe in (
    application_prepared_samples.items()
):
    print(
        sample_name,
        dataframe.shape,
    )

train (215257, 136)
validation (46127, 136)
holdout_test (46127, 136)


In [140]:
# Validação estrutural das amostras
required_modeling_columns = {
    "SK_ID_CURR",
    "TARGET",
}

application_sample_validation = []

for sample_name, dataframe in (
    application_prepared_samples.items()
):
    missing_required_columns = (
        required_modeling_columns
        - set(dataframe.columns)
    )

    application_sample_validation.append(
        {
            "sample": sample_name,
            "n_records": len(dataframe),
            "n_columns": (
                dataframe.shape[1]
            ),
            "n_unique_clients": (
                dataframe[
                    "SK_ID_CURR"
                ]
                .nunique()
            ),
            "n_duplicated_clients": (
                dataframe[
                    "SK_ID_CURR"
                ]
                .duplicated()
                .sum()
            ),
            "n_missing_target": (
                dataframe["TARGET"]
                .isna()
                .sum()
                if "TARGET"
                in dataframe.columns
                else np.nan
            ),
            "missing_required_columns": (
                ", ".join(
                    sorted(
                        missing_required_columns
                    )
                )
            ),
            "event_rate": (
                dataframe["TARGET"].mean()
                if "TARGET"
                in dataframe.columns
                else np.nan
            ),
        }
    )

application_sample_validation = (
    pd.DataFrame(
        application_sample_validation
    )
)

display(
    application_sample_validation.style.format(
        {
            "n_records": "{:,.0f}",
            "n_columns": "{:,.0f}",
            "n_unique_clients": "{:,.0f}",
            "n_duplicated_clients": "{:,.0f}",
            "n_missing_target": "{:,.0f}",
            "event_rate": "{:.2%}",
        }
    )
)

,sample,n_records,n_columns,n_unique_clients,n_duplicated_clients,n_missing_target,missing_required_columns,event_rate
0,train,"215,257",136,"215,257",0,0,,8.07%
1,validation,"46,127",136,"46,127",0,0,,8.07%
2,holdout_test,"46,127",136,"46,127",0,0,,8.07%


In [141]:
# Interrupção se alguma amostra for inválida
if application_sample_validation[
    "n_duplicated_clients"
].gt(0).any():
    raise ValueError(
        "Existem clientes duplicados nas "
        "bases do Notebook 03."
    )

if application_sample_validation[
    "n_missing_target"
].gt(0).any():
    raise ValueError(
        "Existem valores ausentes no TARGET."
    )

if application_sample_validation[
    "missing_required_columns"
].ne("").any():
    raise ValueError(
        "Existem colunas obrigatórias ausentes."
    )

In [142]:
# Validação de sobreposição entre features
application_feature_names = set(
    application_prepared_samples[
        "train"
    ].columns
) - {
    "SK_ID_CURR",
    "TARGET",
}

auxiliary_feature_names = set(
    auxiliary_features_by_client.columns
) - {
    "SK_ID_CURR",
}

overlapping_feature_names = (
    application_feature_names
    & auxiliary_feature_names
)

if overlapping_feature_names:
    raise ValueError(
        "Existem features repetidas entre "
        "application e histórico: "
        f"{sorted(overlapping_feature_names)}"
    )

print(
    "Não existem nomes de features "
    "sobrepostos."
)

Não existem nomes de features sobrepostos.


In [143]:
# Função de merge com as features históricas
def merge_application_with_history(
    application_dataframe,
    auxiliary_dataframe,
):
    original_ids = (
        application_dataframe[
            "SK_ID_CURR"
        ]
        .copy()
    )

    original_target = (
        application_dataframe[
            "TARGET"
        ]
        .copy()
    )

    merged_dataframe = (
        application_dataframe
        .merge(
            auxiliary_dataframe,
            on="SK_ID_CURR",
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    if (
        len(merged_dataframe)
        != len(application_dataframe)
    ):
        raise ValueError(
            "O merge alterou a quantidade "
            "de clientes."
        )

    if not merged_dataframe[
        "SK_ID_CURR"
    ].equals(original_ids):
        raise ValueError(
            "O merge alterou a ordem ou os "
            "identificadores dos clientes."
        )

    if not merged_dataframe[
        "TARGET"
    ].equals(original_target):
        raise ValueError(
            "O merge alterou o TARGET."
        )

    return merged_dataframe

In [144]:
# Combinação das aplicações com o histórico
modeling_samples_full = {
    sample_name: (
        merge_application_with_history(
            application_dataframe=dataframe,
            auxiliary_dataframe=(
                auxiliary_features_by_client
            ),
        )
    )
    for sample_name, dataframe
    in application_prepared_samples.items()
}

for sample_name, dataframe in (
    modeling_samples_full.items()
):
    print(
        sample_name,
        dataframe.shape,
    )

train (215257, 327)
validation (46127, 327)
holdout_test (46127, 327)


In [145]:
# Identificação das flags e contagens históricas
historical_availability_flags = [
    column
    for column in (
        auxiliary_features_by_client.columns
    )
    if (
        column.startswith("HAS_")
        or column
        == "AUX_AVAILABLE_SOURCE_COUNT"
    )
]

historical_count_columns = [
    column
    for column in (
        auxiliary_features_by_client.columns
    )
    if column.endswith("_COUNT")
]

In [146]:
# Preenchimento das flags e contagens
for sample_name, dataframe in (
    modeling_samples_full.items()
):
    dataframe[
        historical_availability_flags
    ] = (
        dataframe[
            historical_availability_flags
        ]
        .fillna(0)
        .astype("int8")
    )

    dataframe[
        historical_count_columns
    ] = (
        dataframe[
            historical_count_columns
        ]
        .fillna(0)
    )

In [147]:
# Construção da cobertura histórica por amostra
historical_coverage_records = []

for sample_name, dataframe in (
    modeling_samples_full.items()
):
    for availability_flag in (
        availability_flag_columns
    ):
        historical_coverage_records.append(
            {
                "sample": sample_name,
                "availability_flag": (
                    availability_flag
                ),
                "n_clients": (
                    dataframe[
                        availability_flag
                    ]
                    .sum()
                ),
                "coverage_rate": (
                    dataframe[
                        availability_flag
                    ]
                    .mean()
                ),
            }
        )

    historical_coverage_records.append(
        {
            "sample": sample_name,
            "availability_flag": (
                "HAS_ANY_AUXILIARY_HISTORY"
            ),
            "n_clients": (
                dataframe[
                    "AUX_AVAILABLE_SOURCE_COUNT"
                ]
                .gt(0)
                .sum()
            ),
            "coverage_rate": (
                dataframe[
                    "AUX_AVAILABLE_SOURCE_COUNT"
                ]
                .gt(0)
                .mean()
            ),
        }
    )

historical_coverage_by_sample = (
    pd.DataFrame(
        historical_coverage_records
    )
)

display(
    historical_coverage_by_sample.style.format(
        {
            "n_clients": "{:,.0f}",
            "coverage_rate": "{:.2%}",
        }
    )
)

,sample,availability_flag,n_clients,coverage_rate
0,train,HAS_BUREAU_HISTORY,"184,421",85.67%
1,train,HAS_PREVIOUS_APPLICATION_HISTORY,"203,801",94.68%
2,train,HAS_INSTALLMENTS_HISTORY,"204,223",94.87%
3,train,HAS_CREDIT_CARD_HISTORY,"61,099",28.38%
4,train,HAS_POS_CASH_HISTORY,"202,687",94.16%
5,train,HAS_ANY_AUXILIARY_HISTORY,"213,735",99.29%
6,validation,HAS_BUREAU_HISTORY,"39,515",85.67%
7,validation,HAS_PREVIOUS_APPLICATION_HISTORY,"43,635",94.60%
8,validation,HAS_INSTALLMENTS_HISTORY,"43,721",94.78%
9,validation,HAS_CREDIT_CARD_HISTORY,"12,996",28.17%


In [148]:
# Identificação das features finais
FINAL_ID_COLUMN = "SK_ID_CURR"
FINAL_TARGET_COLUMN = "TARGET"

final_candidate_features = [
    column
    for column in (
        modeling_samples_full[
            "train"
        ].columns
    )
    if column not in [
        FINAL_ID_COLUMN,
        FINAL_TARGET_COLUMN,
    ]
]

final_train_cardinality = (
    modeling_samples_full[
        "train"
    ][final_candidate_features]
    .nunique(dropna=True)
)

final_all_missing_features = (
    final_train_cardinality[
        final_train_cardinality.eq(0)
    ]
    .index
    .tolist()
)

final_constant_features = (
    final_train_cardinality[
        final_train_cardinality.eq(1)
    ]
    .index
    .tolist()
)

final_unusable_features = sorted(
    set(
        final_all_missing_features
        + final_constant_features
    )
)

print(
    "Features totalmente ausentes em train:",
    len(final_all_missing_features),
)

print(
    "Features constantes em train:",
    len(final_constant_features),
)

Features totalmente ausentes em train: 0
Features constantes em train: 0


In [156]:
# Exibição das features inutilizáveis
final_unusable_feature_table = pd.DataFrame(
    [
        {
            "feature": feature,
            "reason": (
                "Totally missing in train"
                if feature
                in final_all_missing_features
                else "Constant in train"
            ),
        }
        for feature in final_unusable_features
    ],
    columns=[
        "feature",
        "reason",
    ],
)

if final_unusable_feature_table.empty:
    print(
        "Nenhuma feature totalmente ausente "
        "ou constante foi identificada em train."
    )
else:
    display(
        final_unusable_feature_table
    )

Nenhuma feature totalmente ausente ou constante foi identificada em train.


In [150]:
# Remoção das features inutilizáveis
if final_unusable_features:
    for sample_name, dataframe in (
        modeling_samples_full.items()
    ):
        modeling_samples_full[
            sample_name
        ] = dataframe.drop(
            columns=final_unusable_features
        )

In [151]:
# Validação final das bases completas
reference_feature_columns = [
    column
    for column in (
        modeling_samples_full[
            "train"
        ].columns
    )
    if column not in [
        FINAL_ID_COLUMN,
        FINAL_TARGET_COLUMN,
    ]
]

final_modeling_validation = []

for sample_name, dataframe in (
    modeling_samples_full.items()
):
    sample_feature_columns = [
        column
        for column in dataframe.columns
        if column not in [
            FINAL_ID_COLUMN,
            FINAL_TARGET_COLUMN,
        ]
    ]

    numeric_dataframe = (
        dataframe[
            sample_feature_columns
        ]
        .select_dtypes(
            include=[np.number]
        )
    )

    n_infinite_values = (
        np.isinf(
            numeric_dataframe
        )
        .sum()
        .sum()
    )

    final_modeling_validation.append(
        {
            "sample": sample_name,
            "n_records": len(dataframe),
            "n_features": len(
                sample_feature_columns
            ),
            "same_features_as_train": (
                sample_feature_columns
                == reference_feature_columns
            ),
            "n_duplicated_clients": (
                dataframe[
                    FINAL_ID_COLUMN
                ]
                .duplicated()
                .sum()
            ),
            "n_missing_target": (
                dataframe[
                    FINAL_TARGET_COLUMN
                ]
                .isna()
                .sum()
            ),
            "n_infinite_values": (
                n_infinite_values
            ),
            "event_rate": (
                dataframe[
                    FINAL_TARGET_COLUMN
                ]
                .mean()
            ),
        }
    )

final_modeling_validation = pd.DataFrame(
    final_modeling_validation
)

display(
    final_modeling_validation.style.format(
        {
            "n_records": "{:,.0f}",
            "n_features": "{:,.0f}",
            "n_duplicated_clients": "{:,.0f}",
            "n_missing_target": "{:,.0f}",
            "n_infinite_values": "{:,.0f}",
            "event_rate": "{:.2%}",
        }
    )
)

,sample,n_records,n_features,same_features_as_train,n_duplicated_clients,n_missing_target,n_infinite_values,event_rate
0,train,"215,257",325,True,0,0,0,8.07%
1,validation,"46,127",325,True,0,0,0,8.07%
2,holdout_test,"46,127",325,True,0,0,0,8.07%


In [152]:
# Interrupção se alguma base final for inválida
if not final_modeling_validation[
    "same_features_as_train"
].all():
    raise ValueError(
        "As amostras finais possuem "
        "features diferentes."
    )

if final_modeling_validation[
    "n_duplicated_clients"
].gt(0).any():
    raise ValueError(
        "Existem clientes duplicados."
    )

if final_modeling_validation[
    "n_missing_target"
].gt(0).any():
    raise ValueError(
        "Existem valores ausentes no TARGET."
    )

if final_modeling_validation[
    "n_infinite_values"
].gt(0).any():
    raise ValueError(
        "Existem valores infinitos "
        "nas bases finais."
    )

print(
    "Bases finais validadas com sucesso."
)

Bases finais validadas com sucesso.


In [153]:
# Definição dos caminhos das bases finais
final_modeling_output_paths = {
    "train": (
        paths.data_processed
        / "modeling_train_full.parquet"
    ),
    "validation": (
        paths.data_processed
        / "modeling_validation_full.parquet"
    ),
    "holdout_test": (
        paths.data_processed
        / "modeling_holdout_test_full.parquet"
    ),
}

In [154]:
# Persistência das bases finais
for sample_name, dataframe in (
    modeling_samples_full.items()
):
    output_path = (
        final_modeling_output_paths[
            sample_name
        ]
    )

    dataframe.to_parquet(
        output_path,
        index=False,
    )

    print(
        f"{sample_name}: {output_path}"
    )

train: /content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_train_full.parquet
validation: /content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_validation_full.parquet
holdout_test: /content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_holdout_test_full.parquet


In [155]:
# Persistência dos metadados das bases finais
final_modeling_validation.to_csv(
    paths.metadata
    / "final_modeling_validation.csv",
    index=False,
)

historical_coverage_by_sample.to_csv(
    paths.metadata
    / "historical_coverage_by_sample.csv",
    index=False,
)

final_unusable_feature_table.to_csv(
    paths.metadata
    / "final_unusable_features.csv",
    index=False,
)

## Schema Final das Features

O schema final documentará, para cada feature:

- origem;
- tipo de dado;
- classificação numérica ou categórica;
- cardinalidade em `train`;
- proporção de missing em cada amostra;
- indicação de feature histórica ou da aplicação atual.

As características de `validation` e holdout `test` serão registradas apenas para monitoramento. Nenhuma decisão de exclusão ou transformação será ajustada com base nessas amostras.

In [159]:
# Carregamento dos metadados do Notebook 03
application_engineered_metadata_path = (
    paths.metadata
    / "application_engineered_features.csv"
)

if application_engineered_metadata_path.exists():
    application_engineered_metadata = (
        pd.read_csv(
            application_engineered_metadata_path
        )
    )

    application_engineered_features = set(
        application_engineered_metadata[
            "feature"
        ]
        .tolist()
    )
else:
    application_engineered_metadata = (
        pd.DataFrame()
    )

    application_engineered_features = set()

    print(
        "Metadados das features criadas no "
        "Notebook 03 não foram encontrados."
    )

In [160]:
# Função para identificação da origem
def identify_final_feature_source(
    feature,
):
    if feature.startswith("BU_"):
        return "bureau"

    if feature.startswith("PREV_"):
        return "previous_application"

    if feature.startswith("INST_"):
        return "installments_payments"

    if feature.startswith("CC_"):
        return "credit_card_balance"

    if feature.startswith("POS_"):
        return "pos_cash_balance"

    if (
        feature.startswith("HAS_")
        or feature.startswith("AUX_")
    ):
        return "availability"

    return "application"

In [161]:
# Função para classificação do tipo
def classify_final_feature_type(
    series,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return "boolean"

    if pd.api.types.is_numeric_dtype(
        series
    ):
        n_unique = series.nunique(
            dropna=True
        )

        if n_unique <= 2:
            return "binary_numeric"

        if n_unique <= 20:
            return (
                "low_cardinality_numeric"
            )

        return "general_numeric"

    return "categorical"

In [162]:
# Construção do schema final
final_train = (
    modeling_samples_full["train"]
)

final_validation = (
    modeling_samples_full["validation"]
)

final_holdout_test = (
    modeling_samples_full["holdout_test"]
)

final_feature_columns = [
    column
    for column in final_train.columns
    if column not in [
        FINAL_ID_COLUMN,
        FINAL_TARGET_COLUMN,
    ]
]

final_feature_schema_records = []

for feature in final_feature_columns:
    source = (
        identify_final_feature_source(
            feature
        )
    )

    if source != "application":
        feature_role = (
            "historical_engineered"
        )
    elif feature in (
        application_engineered_features
    ):
        feature_role = (
            "application_engineered"
        )
    else:
        feature_role = (
            "application_original"
        )

    final_feature_schema_records.append(
        {
            "feature": feature,
            "source": source,
            "feature_role": feature_role,
            "dtype": str(
                final_train[
                    feature
                ].dtype
            ),
            "feature_type": (
                classify_final_feature_type(
                    final_train[feature]
                )
            ),
            "n_unique_train": (
                final_train[
                    feature
                ]
                .nunique(dropna=True)
            ),
            "missing_rate_train": (
                final_train[
                    feature
                ]
                .isna()
                .mean()
            ),
            "missing_rate_validation": (
                final_validation[
                    feature
                ]
                .isna()
                .mean()
            ),
            "missing_rate_holdout_test": (
                final_holdout_test[
                    feature
                ]
                .isna()
                .mean()
            ),
        }
    )

final_feature_schema = pd.DataFrame(
    final_feature_schema_records
)

final_feature_schema = (
    final_feature_schema
    .sort_values(
        [
            "source",
            "feature",
        ]
    )
    .reset_index(drop=True)
)

display(
    final_feature_schema.style.format(
        {
            "n_unique_train": "{:,.0f}",
            "missing_rate_train": "{:.2%}",
            "missing_rate_validation": "{:.2%}",
            "missing_rate_holdout_test": "{:.2%}",
        }
    )
)

,feature,source,feature_role,dtype,feature_type,n_unique_train,missing_rate_train,missing_rate_validation,missing_rate_holdout_test
0,ADULT_FAMILY_MEMBERS,application,application_engineered,float64,binary_numeric,2,0.00%,0.00%,0.00%
1,AGE_YEARS,application,application_engineered,float64,general_numeric,"17,377",0.00%,0.00%,0.00%
2,AMT_ANNUITY,application,application_original,float64,general_numeric,"12,801",0.00%,0.01%,0.00%
3,AMT_CREDIT,application,application_original,float64,general_numeric,"5,097",0.00%,0.00%,0.00%
4,AMT_GOODS_PRICE,application,application_original,float64,general_numeric,828,0.09%,0.11%,0.08%
5,AMT_INCOME_TOTAL,application,application_original,float64,general_numeric,"1,949",0.00%,0.00%,0.00%
6,AMT_REQ_CREDIT_BUREAU_DAY,application,application_original,float64,low_cardinality_numeric,9,13.51%,13.53%,13.43%
7,AMT_REQ_CREDIT_BUREAU_HOUR,application,application_original,float64,low_cardinality_numeric,5,13.51%,13.53%,13.43%
8,AMT_REQ_CREDIT_BUREAU_MON,application,application_original,float64,general_numeric,22,13.51%,13.53%,13.43%
9,AMT_REQ_CREDIT_BUREAU_QRT,application,application_original,float64,low_cardinality_numeric,10,13.51%,13.53%,13.43%


In [163]:
# Resumo das features por origem
final_features_by_source = (
    final_feature_schema
    .groupby("source")
    .agg(
        n_features=(
            "feature",
            "nunique",
        ),
        n_numeric_features=(
            "feature_type",
            lambda values: (
                values != "categorical"
            ).sum(),
        ),
        n_categorical_features=(
            "feature_type",
            lambda values: (
                values == "categorical"
            ).sum(),
        ),
        mean_missing_rate_train=(
            "missing_rate_train",
            "mean",
        ),
        max_missing_rate_train=(
            "missing_rate_train",
            "max",
        ),
    )
    .reset_index()
)

display(
    final_features_by_source.style.format(
        {
            "n_features": "{:,.0f}",
            "n_numeric_features": "{:,.0f}",
            "n_categorical_features": "{:,.0f}",
            "mean_missing_rate_train": "{:.2%}",
            "max_missing_rate_train": "{:.2%}",
        }
    )
)

,source,n_features,n_numeric_features,n_categorical_features,mean_missing_rate_train,max_missing_rate_train
0,application,134,118,16,22.46%,69.82%
1,availability,6,6,0,0.00%,0.00%
2,bureau,52,52,0,20.17%,74.09%
3,credit_card_balance,35,35,0,56.13%,80.65%
4,installments_payments,25,25,0,3.08%,5.13%
5,pos_cash_balance,26,26,0,12.00%,46.11%
6,previous_application,47,47,0,4.39%,13.17%


In [164]:
# Validação do schema final
if final_feature_schema[
    "feature"
].duplicated().any():
    raise ValueError(
        "Existem features duplicadas "
        "no schema final."
    )

if set(
    final_feature_schema["feature"]
) != set(final_feature_columns):
    raise ValueError(
        "O schema não contempla todas "
        "as features finais."
    )

if final_feature_schema[
    "source"
].eq("unclassified").any():
    raise ValueError(
        "Existem features sem origem."
    )

print(
    "Schema final validado:",
    len(final_feature_schema),
    "features.",
)

Schema final validado: 325 features.


In [165]:
# Persistência do schema final
final_feature_schema_output = (
    paths.metadata
    / "final_feature_schema.csv"
)

final_feature_schema.to_csv(
    final_feature_schema_output,
    index=False,
)

final_features_by_source.to_csv(
    paths.metadata
    / "final_features_by_source.csv",
    index=False,
)

print(
    "Schema salvo em:",
    final_feature_schema_output,
)

Schema salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/final_feature_schema.csv


In [166]:
# Persistência da lista de features em JSON
final_feature_list_output = (
    paths.metadata
    / "final_feature_list.json"
)

with open(
    final_feature_list_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "identifier": (
                FINAL_ID_COLUMN
            ),
            "target": (
                FINAL_TARGET_COLUMN
            ),
            "n_features": len(
                final_feature_columns
            ),
            "features": (
                final_feature_columns
            ),
            "features_by_source": {
                source: (
                    final_feature_schema.loc[
                        final_feature_schema[
                            "source"
                        ].eq(source),
                        "feature",
                    ]
                    .tolist()
                )
                for source in (
                    final_feature_schema[
                        "source"
                    ].unique()
                )
            },
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Lista de features salva em:",
    final_feature_list_output,
)

Lista de features salva em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/final_feature_list.json


## Manifesto das Bases Finais

O manifesto registra as bases que serão utilizadas pelos notebooks de modelagem.

O hash SHA-256 permitirá verificar se algum arquivo foi alterado depois da conclusão deste notebook.

In [167]:
# Função para cálculo do hash SHA-256
import hashlib


def calculate_file_sha256(
    file_path,
    chunk_size=1024 * 1024,
):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [168]:
# Definição dos artefatos finais
final_manifest_artifacts = [
    {
        "artifact": (
            "auxiliary_features_by_client"
        ),
        "sample": "historical_union",
        "file_path": (
            auxiliary_features_output
        ),
        "dataframe": (
            auxiliary_features_by_client
        ),
    },
    {
        "artifact": (
            "modeling_train_full"
        ),
        "sample": "train",
        "file_path": (
            final_modeling_output_paths[
                "train"
            ]
        ),
        "dataframe": (
            modeling_samples_full[
                "train"
            ]
        ),
    },
    {
        "artifact": (
            "modeling_validation_full"
        ),
        "sample": "validation",
        "file_path": (
            final_modeling_output_paths[
                "validation"
            ]
        ),
        "dataframe": (
            modeling_samples_full[
                "validation"
            ]
        ),
    },
    {
        "artifact": (
            "modeling_holdout_test_full"
        ),
        "sample": "holdout_test",
        "file_path": (
            final_modeling_output_paths[
                "holdout_test"
            ]
        ),
        "dataframe": (
            modeling_samples_full[
                "holdout_test"
            ]
        ),
    },
]

In [169]:
# Construção do manifesto final
manifest_generated_at = (
    datetime.now(timezone.utc)
    .isoformat()
)

final_manifest_records = []

for artifact in final_manifest_artifacts:
    file_path = Path(
        artifact["file_path"]
    )

    dataframe = artifact[
        "dataframe"
    ]

    if not file_path.exists():
        raise FileNotFoundError(
            f"Artefato ausente: {file_path}"
        )

    event_rate = (
        dataframe[
            FINAL_TARGET_COLUMN
        ]
        .mean()
        if FINAL_TARGET_COLUMN
        in dataframe.columns
        else np.nan
    )

    final_manifest_records.append(
        {
            "artifact": (
                artifact["artifact"]
            ),
            "sample": (
                artifact["sample"]
            ),
            "file_name": (
                file_path.name
            ),
            "file_path": str(
                file_path
            ),
            "n_records": len(
                dataframe
            ),
            "n_columns": (
                dataframe.shape[1]
            ),
            "n_features": (
                dataframe.shape[1]
                - (
                    2
                    if FINAL_TARGET_COLUMN
                    in dataframe.columns
                    else 1
                )
            ),
            "event_rate": event_rate,
            "file_size_mb": (
                file_path.stat().st_size
                / 1024**2
            ),
            "sha256": (
                calculate_file_sha256(
                    file_path
                )
            ),
            "generated_at_utc": (
                manifest_generated_at
            ),
        }
    )

final_data_manifest = pd.DataFrame(
    final_manifest_records
)

display(
    final_data_manifest.style.format(
        {
            "n_records": "{:,.0f}",
            "n_columns": "{:,.0f}",
            "n_features": "{:,.0f}",
            "event_rate": "{:.2%}",
            "file_size_mb": "{:,.2f}",
        }
    )
)

,artifact,sample,file_name,file_path,n_records,n_columns,n_features,event_rate,file_size_mb,sha256,generated_at_utc
0,auxiliary_features_by_client,historical_union,auxiliary_features_by_client.parquet,/content/drive/MyDrive/home-credit-default-risk/data/interim/auxiliary_features_by_client.parquet,"353,895",192,191,nan%,95.19,d4408997d358900242bd8d38dd90e35673d89f965068dd461d628069d064829a,2026-10-01T02:44:47.599083+00:00
1,modeling_train_full,train,modeling_train_full.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_train_full.parquet,"215,257",327,325,8.07%,84.05,c1e84997831c09739d5ffd197b54ded4697ffdc311885a2e482e9a9ebeac36cb,2026-10-01T02:44:47.599083+00:00
2,modeling_validation_full,validation,modeling_validation_full.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_validation_full.parquet,"46,127",327,325,8.07%,19.61,bd955b506ec0c0ef3ad28f4ff9f6c6d64fe7080765fa07f97311ce7e09763063,2026-10-01T02:44:47.599083+00:00
3,modeling_holdout_test_full,holdout_test,modeling_holdout_test_full.parquet,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_holdout_test_full.parquet,"46,127",327,325,8.07%,19.61,e7c11ed77d9a5ab6105a7cef7fb2eaf9e8b34511e9174f8314097df357814285,2026-10-01T02:44:47.599083+00:00


In [170]:
# Validação do manifesto final
final_manifest_validation = pd.DataFrame(
    {
        "validation": [
            "Hashes possuem 64 caracteres",
            "Artefatos não estão duplicados",
            "Arquivos possuem registros",
            (
                "Amostras supervisionadas "
                "possuem o mesmo número de features"
            ),
        ],
        "result": [
            final_data_manifest[
                "sha256"
            ].str.len().eq(64).all(),

            not final_data_manifest[
                "artifact"
            ].duplicated().any(),

            final_data_manifest[
                "n_records"
            ].gt(0).all(),

            final_data_manifest.loc[
                final_data_manifest[
                    "sample"
                ].isin(
                    [
                        "train",
                        "validation",
                        "holdout_test",
                    ]
                ),
                "n_features",
            ].nunique() == 1,
        ],
    }
)

display(final_manifest_validation)

,validation,result
0,Hashes possuem 64 caracteres,True
1,Artefatos não estão duplicados,True
2,Arquivos possuem registros,True
3,Amostras supervisionadas possuem o mesmo númer...,True


In [171]:
# Interrupção se o manifesto for inválido
if not final_manifest_validation[
    "result"
].all():
    failed_validations = (
        final_manifest_validation.loc[
            ~final_manifest_validation[
                "result"
            ],
            "validation",
        ]
        .tolist()
    )

    raise ValueError(
        "Falhas no manifesto final: "
        f"{failed_validations}"
    )

print(
    "Manifesto final validado."
)

Manifesto final validado.


In [172]:
# Persistência do manifesto final em CSV
final_data_manifest_output = (
    paths.metadata
    / "final_data_manifest.csv"
)

final_data_manifest.to_csv(
    final_data_manifest_output,
    index=False,
)

In [173]:
# Persistência do manifesto final em JSON
final_manifest_json_records = (
    final_data_manifest
    .astype(object)
    .where(
        pd.notna(
            final_data_manifest
        ),
        None,
    )
    .to_dict(
        orient="records"
    )
)

final_data_manifest_json_output = (
    paths.metadata
    / "final_data_manifest.json"
)

with open(
    final_data_manifest_json_output,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "generated_at_utc": (
                manifest_generated_at
            ),
            "hash_algorithm": "SHA-256",
            "artifacts": (
                final_manifest_json_records
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Manifesto CSV salvo em:",
    final_data_manifest_output,
)

print(
    "Manifesto JSON salvo em:",
    final_data_manifest_json_output,
)

Manifesto CSV salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/final_data_manifest.csv
Manifesto JSON salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/final_data_manifest.json


## Consolidação dos Resultados

O notebook transformou seis tabelas relacionais do Home Credit em features históricas com uma linha por cliente.

O processamento foi realizado em níveis sucessivos:

- registros mensais para contratos;
- pagamentos para parcelas;
- parcelas para contratos;
- contratos para clientes;
- fontes históricas para uma tabela consolidada;
- histórico consolidado para as amostras de modelagem.

As agregações foram construídas sem utilização do `TARGET`. A variável resposta foi incorporada somente por meio das bases supervisionadas produzidas no Notebook 03.

In [174]:
# Construção do resumo quantitativo
notebook_04_summary = pd.DataFrame(
    [
        {
            "indicator": (
                "Tabelas auxiliares processadas"
            ),
            "value": len(
                AUXILIARY_TABLE_SPECS
            ),
        },
        {
            "indicator": (
                "Clientes na união histórica"
            ),
            "value": len(
                auxiliary_features_by_client
            ),
        },
        {
            "indicator": (
                "Features históricas consolidadas"
            ),
            "value": (
                auxiliary_features_by_client
                .shape[1]
                - 1
            ),
        },
        {
            "indicator": (
                "Features finais de modelagem"
            ),
            "value": len(
                final_feature_columns
            ),
        },
        {
            "indicator": (
                "Registros finais em train"
            ),
            "value": len(
                modeling_samples_full[
                    "train"
                ]
            ),
        },
        {
            "indicator": (
                "Registros finais em validation"
            ),
            "value": len(
                modeling_samples_full[
                    "validation"
                ]
            ),
        },
        {
            "indicator": (
                "Registros finais no holdout test"
            ),
            "value": len(
                modeling_samples_full[
                    "holdout_test"
                ]
            ),
        },
        {
            "indicator": (
                "Features totalmente ausentes "
                "em train"
            ),
            "value": len(
                final_all_missing_features
            ),
        },
        {
            "indicator": (
                "Features constantes em train"
            ),
            "value": len(
                final_constant_features
            ),
        },
    ]
)

display(
    notebook_04_summary.style.format(
        {
            "value": "{:,.0f}",
        }
    )
)

,indicator,value
0,Tabelas auxiliares processadas,6
1,Clientes na união histórica,"353,895"
2,Features históricas consolidadas,191
3,Features finais de modelagem,325
4,Registros finais em train,"215,257"
5,Registros finais em validation,"46,127"
6,Registros finais no holdout test,"46,127"
7,Features totalmente ausentes em train,0
8,Features constantes em train,0


In [175]:
# Consolidação do resumo por fonte
source_summary = (
    final_features_by_source
    .merge(
        auxiliary_source_coverage[
            [
                "source",
                "n_clients",
                (
                    "proportion_of_"
                    "historical_union"
                ),
            ]
        ],
        on="source",
        how="left",
    )
)

display(
    source_summary.style.format(
        {
            "n_features": "{:,.0f}",
            "n_numeric_features": "{:,.0f}",
            "n_categorical_features": "{:,.0f}",
            "mean_missing_rate_train": "{:.2%}",
            "max_missing_rate_train": "{:.2%}",
            "n_clients": "{:,.0f}",
            (
                "proportion_of_"
                "historical_union"
            ): "{:.2%}",
        }
    )
)

,source,n_features,n_numeric_features,n_categorical_features,mean_missing_rate_train,max_missing_rate_train,n_clients,proportion_of_historical_union
0,application,134,118,16,22.46%,69.82%,nan,nan%
1,availability,6,6,0,0.00%,0.00%,nan,nan%
2,bureau,52,52,0,20.17%,74.09%,"305,811",86.41%
3,credit_card_balance,35,35,0,56.13%,80.65%,nan,nan%
4,installments_payments,25,25,0,3.08%,5.13%,nan,nan%
5,pos_cash_balance,26,26,0,12.00%,46.11%,nan,nan%
6,previous_application,47,47,0,4.39%,13.17%,"338,857",95.75%


In [176]:
# Definição das entregas obrigatórias
notebook_04_required_outputs = {
    "bureau_by_client": (
        paths.data_interim
        / "bureau_by_client.parquet"
    ),
    "previous_application_by_client": (
        paths.data_interim
        / "previous_application_by_client.parquet"
    ),
    "installments_by_client": (
        paths.data_interim
        / "installments_by_client.parquet"
    ),
    "credit_card_by_client": (
        paths.data_interim
        / "credit_card_by_client.parquet"
    ),
    "pos_cash_by_client": (
        paths.data_interim
        / "pos_cash_by_client.parquet"
    ),
    "auxiliary_features_by_client": (
        paths.data_interim
        / "auxiliary_features_by_client.parquet"
    ),
    "modeling_train_full": (
        paths.data_processed
        / "modeling_train_full.parquet"
    ),
    "modeling_validation_full": (
        paths.data_processed
        / "modeling_validation_full.parquet"
    ),
    "modeling_holdout_test_full": (
        paths.data_processed
        / "modeling_holdout_test_full.parquet"
    ),
    "final_feature_schema": (
        paths.metadata
        / "final_feature_schema.csv"
    ),
    "final_feature_list": (
        paths.metadata
        / "final_feature_list.json"
    ),
    "final_data_manifest": (
        paths.metadata
        / "final_data_manifest.csv"
    ),
}

In [177]:
# Validação das entregas obrigatórias
notebook_04_output_validation = (
    pd.DataFrame(
        [
            {
                "artifact": artifact,
                "file_path": str(
                    file_path
                ),
                "exists": (
                    file_path.exists()
                ),
                "file_size_mb": (
                    file_path.stat().st_size
                    / 1024**2
                    if file_path.exists()
                    else np.nan
                ),
            }
            for artifact, file_path
            in (
                notebook_04_required_outputs
                .items()
            )
        ]
    )
)

display(
    notebook_04_output_validation.style.format(
        {
            "file_size_mb": "{:,.2f}",
        }
    )
)

,artifact,file_path,exists,file_size_mb
0,bureau_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/bureau_by_client.parquet,True,28.04
1,previous_application_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/previous_application_by_client.parquet,True,35.88
2,installments_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/installments_by_client.parquet,True,13.29
3,credit_card_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/credit_card_by_client.parquet,True,8.58
4,pos_cash_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/pos_cash_by_client.parquet,True,9.69
5,auxiliary_features_by_client,/content/drive/MyDrive/home-credit-default-risk/data/interim/auxiliary_features_by_client.parquet,True,95.19
6,modeling_train_full,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_train_full.parquet,True,84.05
7,modeling_validation_full,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_validation_full.parquet,True,19.61
8,modeling_holdout_test_full,/content/drive/MyDrive/home-credit-default-risk/data/processed/modeling_holdout_test_full.parquet,True,19.61
9,final_feature_schema,/content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/final_feature_schema.csv,True,0.04


In [178]:
# Interrupção se alguma entrega estiver ausente
if not notebook_04_output_validation[
    "exists"
].all():
    missing_outputs = (
        notebook_04_output_validation.loc[
            ~notebook_04_output_validation[
                "exists"
            ],
            "artifact",
        ]
        .tolist()
    )

    raise FileNotFoundError(
        "Entregas obrigatórias ausentes: "
        f"{missing_outputs}"
    )

print(
    "Todas as entregas obrigatórias "
    "foram encontradas."
)

Todas as entregas obrigatórias foram encontradas.


In [179]:
# Persistência do resumo do Notebook 04
notebook_04_summary_output = (
    paths.metadata
    / "notebook_04_summary.csv"
)

notebook_04_summary.to_csv(
    notebook_04_summary_output,
    index=False,
)

source_summary.to_csv(
    paths.metadata
    / "notebook_04_source_summary.csv",
    index=False,
)

print(
    "Resumo salvo em:",
    notebook_04_summary_output,
)

Resumo salvo em: /content/drive/MyDrive/home-credit-default-risk/artifacts/metadata/notebook_04_summary.csv


## Limitações

As features históricas possuem limitações que devem ser consideradas na interpretação dos modelos.

### Disponibilidade temporal

As agregações utilizaram variáveis temporais relativas à aplicação atual e excluíram movimentos explicitamente posteriores quando identificáveis.

Entretanto, a base não fornece um catálogo completo de disponibilidade temporal de cada campo. Portanto, a ausência de data leakage foi avaliada com base na semântica e nos campos temporais disponíveis.

### Datas futuras previstas

Algumas datas positivas, como vencimentos contratuais previstos, podem representar informações futuras já conhecidas no momento da aplicação. Esses valores não foram automaticamente classificados como leakage.

### Agregações

A transformação para uma linha por cliente resume sequências e pode perder detalhes da trajetória temporal.

Médias calculadas entre contratos atribuem o mesmo peso a contratos com diferentes quantidades de observações. Para reduzir essa limitação, também foram mantidas contagens, somas, máximos e algumas razões calculadas sobre totais.

### Ausência de histórico

A inexistência de registros em uma fonte pode representar tanto ausência de relacionamento quanto indisponibilidade da informação. Flags de disponibilidade foram criadas para permitir que os modelos diferenciem esses casos.

### Valores ausentes

Médias, máximos, valores e razões permaneceram como missing quando não havia informação suficiente. A imputação será definida nos pipelines específicos.

### Janelas temporais

Foram criadas janelas recentes de seis meses para POS/CASH e cartão de crédito. Outras janelas, como três, doze ou vinte e quatro meses, poderão ser avaliadas posteriormente caso apresentem ganho de desempenho e estabilidade.

### Validação temporal

As amostras de `train`, `validation` e holdout `test` continuam sendo divisões aleatórias estratificadas. As features históricas não transformam essa estratégia em validação `Out-of-Time`.

## Conclusão

O notebook consolidou o histórico financeiro dos clientes a partir de seis tabelas auxiliares do Home Credit.

As features produzidas representam diferentes dimensões do risco:

- exposição e endividamento em outras instituições;
- aplicações anteriores;
- aprovação e recusa de propostas;
- pontualidade e suficiência dos pagamentos;
- utilização de cartão de crédito;
- atrasos em contratos POS/CASH;
- disponibilidade das diferentes fontes de histórico.

As agregações foram combinadas com as amostras definidas anteriormente sem alterar clientes, identificadores ou `TARGET`.

As bases finais possuem o mesmo conjunto de features e estão prontas para a construção dos pipelines de modelagem.

Nenhuma imputação, seleção supervisionada, WoE, encoding ou scaling foi realizada neste notebook. Esses procedimentos serão ajustados exclusivamente em `train` dentro de cada abordagem.